In [1]:
!pip install -q dm-haiku optax

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 377.0/377.0 kB 7.7 MB/s eta 0:00:00:00:01


In [2]:
# Cell 0 (اختیاری، قبل از همه)
import os
os.environ["RUN_MODE"] = "smoke"        # "smoke" -> "medium" -> "full"
os.environ["OUT_DIR"]  = "/kaggle/working/v18"

In [3]:
# =============================================================================
# CELL 1 / 9  --  CONFIGURATION  (V18.0, single source of truth)
# =============================================================================
# Design rules enforced by this cell:
#   R1  Every constant used anywhere downstream lives HERE. No cell may define
#       a physical constant locally. Cell 9 asserts this by hashing CFG.
#   R2  The reference (ground-truth) solver is calibrated to a REALISTIC
#       dynamic range (Validity Boundary 25). CFG asserts it after Cell 2 runs.
#   R3  RUN_MODE only scales COST (epochs, batch, pool sizes). It never changes
#       physics, architecture or metrics. A smoke run and a full run must be
#       comparable in kind.
# =============================================================================

import os, json, hashlib, dataclasses
from dataclasses import dataclass, field, asdict
from typing import Tuple

import numpy as np
import jax

# float64 everywhere. Required for (a) the FD reference solver, (b) scipy
# L-BFGS-B, which is numerically meaningless in float32 for this problem.
jax.config.update("jax_enable_x64", True)

RUN_MODE = os.environ.get("RUN_MODE", "medium")     # "smoke" | "medium" | "full"
assert RUN_MODE in ("smoke", "medium", "full")

OUT_DIR = os.environ.get("OUT_DIR", "/kaggle/working/v18")
os.makedirs(OUT_DIR, exist_ok=True)


# -----------------------------------------------------------------------------
# 1. Physical / domain constants
# -----------------------------------------------------------------------------
@dataclass(frozen=True)
class Domain:
    nx: int = 100               # cells along-strait  (x)
    ny: int = 50                # cells across-strait (y)
    dx: float = 1000.0          # m
    dy: float = 1000.0          # m

    @property
    def Lx(self) -> float: return self.nx * self.dx
    @property
    def Ly(self) -> float: return self.ny * self.dy


@dataclass(frozen=True)
class Physics:
    g: float = 9.81
    rho_w: float = 1025.0
    rho_a: float = 1.225
    lat_deg: float = 26.5
    omega_earth: float = 7.2921e-5

    # Bathymetry (bed elevation b < 0; still-water depth H = -b)
    depth_min: float = 25.0     # m, mud flats near Qeshm
    depth_max: float = 100.0    # m, mid-channel
    ridge_amp: float = 12.0     # m, rocky sills superposed on the channel

    # True bed friction field: two smooth zones (rock ridge vs mud flat)
    cf_rock: float = 0.0070
    cf_mud:  float = 0.0020
    cf_transition_cells: float = 3.0

    # Sigmoid bounds of the friction branch (must strictly bracket the truth)
    cf_lo: float = 0.0015
    cf_hi: float = 0.0080

    # Tidal forcing at the eastern open boundary (Gulf of Oman side).
    # Calibrated so that peak |u| lands in 0.4-1.0 m/s and the cycle-mean
    # exchange lands near the observed 0.15 m/s (Johns et al., 2003).
    # THIS IS THE FIX FOR VALIDITY BOUNDARY 25.
    a_m2: float = 0.90          # m
    a_s2: float = 0.30          # m
    T_m2: float = 44712.0       # s (12.42 h)
    T_s2: float = 43200.0       # s (12.00 h)
    phase_s2: float = 0.7       # rad

    # Wind: Large & Pond (1981) drag, spatially varying, seasonally modulated
    wind_speed: float = 8.0     # m/s reference magnitude
    wind_dir_deg: float = 315.0 # meteorological "from" direction (NW shamal)
    wind_spatial_amp: float = 0.35   # relative across-strait modulation
    wind_seasonal_amp: float = 0.25  # relative modulation over the run

    # Adaptive eddy viscosity nu(x,y) from bed-slope proxy.
    # nu_max raised from the V17 value: the V17 ablation showed the diffusion
    # term changed the trained weights by only 4e-5 relative, i.e. it was
    # numerically inert. Cell 3 now ASSERTS that std(nu) > 0 when enabled.
    nu_base: float = 5.0        # m^2/s
    nu_max: float = 80.0        # m^2/s
    nu_sigmoid_k: float = 6.0

    # Numerical viscosity of the FD reference solver (kept small; the upwind
    # advection scheme provides the stabilisation, not this term)
    nu_fd: float = 40.0         # m^2/s

    @property
    def f_coriolis(self) -> float:
        return 2.0 * self.omega_earth * np.sin(np.deg2rad(self.lat_deg))

    @property
    def omega_m2(self) -> float: return 2.0 * np.pi / self.T_m2
    @property
    def omega_s2(self) -> float: return 2.0 * np.pi / self.T_s2


@dataclass(frozen=True)
class RefSolver:
    """Arakawa C-grid, explicit, first-order upwind momentum advection.

    The move from the V17 collocated A-grid to a staggered C-grid does two
    things at once:
      (a) removes the pressure-velocity coupling mode of Validity Boundary 16,
          so the true Cf field no longer has to be artificially smoothed;
      (b) strengthens the anti-inverse-crime argument: the reference solver and
          the PINN no longer share a variable arrangement, only the continuum
          equations.
    """
    dt: float = 5.0             # s   (CFL: dx/sqrt(g*Hmax) ~ 32 s, so ~6x margin)
    n_tidal_cycles: float = 3.0
    warmup_cycles: float = 1.0  # discarded from every reported metric
    n_snapshots: int = 60       # saved AFTER warmup
    ramp_cycles: float = 0.5    # smooth spin-up of the boundary forcing


@dataclass(frozen=True)
class Scales:
    """Non-dimensionalisation. Chosen so every PDE coefficient is O(0.1-20)."""
    H_eta: float = 1.0          # m,   free-surface scale
    U: float = 0.30             # m/s, velocity scale
    H_D: float = 60.0           # m,   mean water depth scale
    T: float = 44712.0          # s,   M2 period


@dataclass(frozen=True)
class Arch:
    fourier_features: int = 64
    sigma_hydro: float = 1.0
    # sigma_fric is swept in Cell 9; this is the centre of the sweep.
    sigma_fric: float = 2.0
    hydro_layers: Tuple[int, ...] = (128, 128, 128, 128)
    fric_layers: Tuple[int, ...] = (128, 128, 128)
    y_aspect_rescale: bool = True   # compensate the 2:1 strait aspect ratio

    # Friction output parameterisation.
    #   "log_sigmoid" : Cf = exp(log(lo) + (log(hi)-log(lo)) * sigmoid(z/temp))
    # Log-space keeps relative resolution uniform across the band and, together
    # with temp > 1, keeps the pre-activation away from the flat tails.
    # V17 used a linear sigmoid and ended training ~93% saturated
    # (sigmoid_deriv_mean 0.018 of a 0.25 maximum) -> Validity Boundary 6 was
    # ACTIVE, not mitigated. This is the fix.
    fric_output: str = "log_sigmoid"
    fric_sigmoid_temp: float = 2.0
    fric_saturation_penalty: float = 1e-4   # weak L2 on the pre-activation


@dataclass(frozen=True)
class LossCfg:
    # Base weights. Cell 6 additionally applies per-term gradient-norm
    # balancing, so these are priors, not final magnitudes.
    w_data: Tuple[float, float] = (1.0, 1.0)    # (phase1, phase2)
    w_swe: Tuple[float, float] = (0.1, 1.0)
    w_tv: Tuple[float, float] = (0.0, 1e-3)
    w_bc: float = 1.0

    # Gradient-norm balancing. V17's end-of-training loss was ~95% TV
    # (TV=9.9e-3 vs total=5.8e-4 with data=5e-6, physics=7.4e-6) while TV was
    # simultaneously INCREASING. Balancing by gradient norm rather than by loss
    # value is the only way to see and control that.
    grad_balance: bool = True
    grad_balance_every: int = 100
    grad_balance_beta: float = 0.9
    tv_max_grad_share: float = 0.15   # TV may never exceed 15% of total grad norm

    # Causal weighting. V17 used a static w(t)=exp(-eps*t), which merely
    # down-weights late times. This implements the actual Wang, Sankaran &
    # Perdikaris (2024) scheme: chunk weights driven by the cumulative
    # upstream residual, recomputed under stop_gradient.
    causal_chunks: int = 16
    causal_eps: float = 1.0
    causal_enabled: bool = True

    huber_delta: float = 1.0


@dataclass(frozen=True)
class OptCfg:
    lr_hydro: float = 1e-3
    lr_fric: float = 5e-4
    lr_decay_to: float = 0.05     # cosine floor as a fraction of peak
    grad_clip: float = 1.0

    # Freeze-release. The friction learning rate ramps 0 -> lr_fric linearly
    # and the friction optimiser moment state is reset exactly at the boundary.
    freeze_release: bool = True

    # L-BFGS-B. V17 reported nit=1000 / converged=False in EVERY run while the
    # proposal text claimed "natural convergence, not an iteration cap". Here
    # the cap is generous, the tolerances are explicit, and Cell 7 records the
    # scipy status verbatim.
    lbfgs_enabled: bool = True
    lbfgs_maxiter: int = 20000
    lbfgs_maxfun: int = 25000
    lbfgs_ftol: float = 1e-14
    lbfgs_gtol: float = 1e-12
    # Accept the L-BFGS result ONLY if RMSE(Cf) against truth did not worsen.
    lbfgs_revert_if_cf_worse: bool = True
    lbfgs_revert_tolerance_pp: float = 0.5   # percentage points of slack


# -----------------------------------------------------------------------------
# 2. Cost profile: the ONLY thing RUN_MODE controls
# -----------------------------------------------------------------------------
# V17 ran 6000 epochs with a 1000-point physics batch over a 360k pool
#   -> effective_passes_physics = 16.67, against a reference floor of 20.
# The friction branch was therefore under-trained by more than an order of
# magnitude. "medium" below gives ~670 effective passes.
_COST = {
    "smoke":  dict(epochs=3000,  phase1_epochs=1200, warm_epochs=200,
                   batch_data=2000, batch_phys=2000,
                   pool_phys=60_000, n_obs_tracks=6,  lbfgs_maxiter=500),
    "medium": dict(epochs=30_000, phase1_epochs=10_000, warm_epochs=1500,
                   batch_data=8000, batch_phys=8000,
                   pool_phys=360_000, n_obs_tracks=10, lbfgs_maxiter=5000),
    "full":   dict(epochs=120_000, phase1_epochs=35_000, warm_epochs=5000,
                   batch_data=16_000, batch_phys=20_000,
                   pool_phys=1_200_000, n_obs_tracks=14, lbfgs_maxiter=20_000),
}


@dataclass(frozen=True)
class Cost:
    epochs: int
    phase1_epochs: int
    warm_epochs: int
    batch_data: int
    batch_phys: int
    pool_phys: int
    n_obs_tracks: int
    lbfgs_maxiter: int
    log_every: int = 50
    eval_every: int = 1000

    @property
    def effective_passes_physics(self) -> float:
        return self.epochs * self.batch_phys / self.pool_phys


@dataclass(frozen=True)
class ObsCfg:
    noise_pct: float = 3.0          # Gaussian, relative to the eta scale
    track_angle_deg: float = 72.0   # ground-track inclination proxy
    along_track_spacing_m: float = 6000.0
    revisit_snapshots: int = 5      # a track is only sampled every k snapshots
    # Diagnostic arm: if True, u and v are observed as well as h. This is the
    # single most informative experiment available, because it separates a
    # STRUCTURAL identifiability limit (Radfar 2026) from an optimisation
    # failure. It is not part of the scientific claim; it is a control.
    observe_velocity: bool = False


@dataclass(frozen=True)
class Config:
    domain: Domain = field(default_factory=Domain)
    phys: Physics = field(default_factory=Physics)
    ref: RefSolver = field(default_factory=RefSolver)
    scl: Scales = field(default_factory=Scales)
    arch: Arch = field(default_factory=Arch)
    loss: LossCfg = field(default_factory=LossCfg)
    opt: OptCfg = field(default_factory=OptCfg)
    obs: ObsCfg = field(default_factory=ObsCfg)
    cost: Cost = field(default_factory=lambda: Cost(**_COST[RUN_MODE]))

    run_mode: str = RUN_MODE
    seed: int = 0
    wind_enabled: bool = True
    diffusion_enabled: bool = True

    # ---- derived PDE coefficients (non-dimensional) ----------------------
    # The network inputs are normalised to [-1, 1] in t, x, y. A unit step in
    # the normalised coordinate therefore spans HALF the physical extent, so
    # every length/time scale below is a half-span. Getting this wrong by a
    # factor of 2 silently rescales the inferred Cf, so it is derived here once
    # and asserted in Cell 3 against a finite-difference check on the reference
    # fields.
    @property
    def Lx_eff(self) -> float: return self.domain.Lx / 2.0
    @property
    def Ly_eff(self) -> float: return self.domain.Ly / 2.0
    @property
    def T_eff(self) -> float:
        window = (self.ref.n_tidal_cycles - self.ref.warmup_cycles) * self.phys.T_m2
        return window / 2.0

    def pde_coeffs(self) -> dict:
        p, s = self.phys, self.scl
        Lx, Ly, T = self.Lx_eff, self.Ly_eff, self.T_eff
        return dict(
            # continuity:  eta_t + a1*(D u)_x + a2*(D v)_y = 0
            a1=T * s.H_D * s.U / (s.H_eta * Lx),
            a2=T * s.H_D * s.U / (s.H_eta * Ly),
            # momentum
            c_adv_x=s.U * T / Lx,
            c_adv_y=s.U * T / Ly,
            c_grad_x=p.g * s.H_eta * T / (Lx * s.U),
            c_grad_y=p.g * s.H_eta * T / (Ly * s.U),
            c_cor=p.f_coriolis * T,
            c_fric=s.U * T / s.H_D,
            c_wind=T / (p.rho_w * s.H_D * s.U),
            c_diff_x=T / (Lx ** 2),
            c_diff_y=T / (Ly ** 2),
            D_floor=0.05,   # non-dimensional, i.e. 3 m of water
        )

    def fingerprint(self) -> str:
        blob = json.dumps(asdict(self), sort_keys=True, default=str)
        return hashlib.sha256(blob.encode()).hexdigest()[:12]

    def tag(self) -> str:
        return (f"wind_{'on' if self.wind_enabled else 'off'}"
                f"_diff_{'on' if self.diffusion_enabled else 'off'}"
                f"{'_obsuv' if self.obs.observe_velocity else ''}"
                f"{'_nofreeze' if not self.opt.freeze_release else ''}"
                f"_{self.run_mode}")


CFG = Config()

# -----------------------------------------------------------------------------
# 3. Self-checks that would have caught three of the V17 failures
# -----------------------------------------------------------------------------
def audit_config(cfg: Config = CFG, verbose: bool = True) -> dict:
    d, p, r, s, c = cfg.domain, cfg.phys, cfg.ref, cfg.scl, cfg.cost
    rep = {}

    cfl = d.dx / np.sqrt(p.g * (p.depth_max + p.a_m2 + p.a_s2))
    rep["cfl_limit_s"] = cfl
    rep["cfl_ratio"] = r.dt / cfl
    assert r.dt < 0.5 * cfl, f"dt={r.dt}s violates CFL (limit {cfl:.1f}s)"

    rep["effective_passes_physics"] = c.effective_passes_physics
    assert c.effective_passes_physics >= 20.0, (
        f"physics pool would be traversed only {c.effective_passes_physics:.1f} "
        "times; V17 failed at 16.7. Raise epochs or batch_phys.")

    assert p.cf_lo < p.cf_mud < p.cf_rock < p.cf_hi, \
        "sigmoid bounds must strictly bracket the true Cf range"
    rep["cf_band_margin_lo"] = p.cf_mud / p.cf_lo
    rep["cf_band_margin_hi"] = p.cf_hi / p.cf_rock

    if cfg.diffusion_enabled:
        assert p.nu_max > 4.0 * p.nu_base, \
            "nu_max/nu_base too close: the diffusion ablation would be inert"

    rep["n_steps_total"] = int(r.n_tidal_cycles * p.T_m2 / r.dt)
    rep["sim_hours"] = r.n_tidal_cycles * p.T_m2 / 3600.0
    rep["pde_coeffs"] = cfg.pde_coeffs()
    rep["fingerprint"] = cfg.fingerprint()

    bad = {k: v for k, v in rep["pde_coeffs"].items()
           if isinstance(v, float) and v != 0 and not (1e-6 < abs(v) < 1e4)}
    assert not bad, f"badly scaled PDE coefficients: {bad}"

    # ---- a priori magnitude audit of the adaptive-diffusion term ----------
    # This is the check that would have explained the V17 result (diffusion
    # ON vs OFF changed the trained weights by 4e-5 relative) WITHOUT burning
    # 75% extra L-BFGS time to find out. A horizontal eddy viscosity of
    # O(10-100) m^2/s acting over a 25 km half-span has a diffusive timescale
    # of L^2/nu ~ 10^7 s, i.e. ~100 days, against a 12 h tidal timescale.
    # The term is therefore intrinsically 3-4 orders of magnitude weaker than
    # pressure gradient and friction. That is PHYSICS, not a bug.
    co = rep["pde_coeffs"]
    diff_mag = max(co["c_diff_x"], co["c_diff_y"]) * p.nu_max
    ref_mag = max(co["c_grad_x"], co["c_fric"] * p.cf_rock)
    rep["diffusion_relative_magnitude"] = diff_mag / ref_mag
    if cfg.diffusion_enabled and rep["diffusion_relative_magnitude"] < 1e-2:
        print("[CFG] WARNING: adaptive diffusion is ~%.1e of the leading "
              "momentum terms.\n"
              "       Expect it to be numerically undetectable, exactly as in "
              "V17. Either\n"
              "       (a) report it as a negative result and drop the novelty "
              "claim, or\n"
              "       (b) justify a much larger nu_max on physical grounds, or\n"
              "       (c) restrict the claim to a sub-domain with steep bed "
              "gradients." % rep["diffusion_relative_magnitude"])

    if verbose:
        print(f"[CFG] mode={cfg.run_mode}  tag={cfg.tag()}  fp={rep['fingerprint']}")
        print(f"[CFG] dt={r.dt}s  CFL limit={cfl:.1f}s  ratio={rep['cfl_ratio']:.3f}")
        print(f"[CFG] reference run: {rep['sim_hours']:.1f} h in "
              f"{rep['n_steps_total']:,} steps")
        print(f"[CFG] effective physics passes = "
              f"{rep['effective_passes_physics']:.1f}  (V17 was 16.7)")
        print(f"[CFG] PDE coefficients: " + "  ".join(
            f"{k}={v:.3g}" for k, v in rep["pde_coeffs"].items()))
    return rep


def save_config(cfg: Config = CFG, path: str = None) -> str:
    path = path or os.path.join(OUT_DIR, f"config_{cfg.tag()}.json")
    with open(path, "w") as fh:
        json.dump({"config": asdict(cfg),
                   "fingerprint": cfg.fingerprint(),
                   "pde_coeffs": cfg.pde_coeffs()}, fh, indent=2, default=str)
    return path


def with_overrides(**kw) -> Config:
    """Return a copy of CFG with top-level fields replaced (used by Cell 9)."""
    return dataclasses.replace(CFG, **kw)


AUDIT = audit_config()
print(f"[CFG] saved -> {save_config()}")

[CFG] WARNING: adaptive diffusion is ~2.0e-04 of the leading momentum terms.
       Expect it to be numerically undetectable, exactly as in V17. Either
       (a) report it as a negative result and drop the novelty claim, or
       (b) justify a much larger nu_max on physical grounds, or
       (c) restrict the claim to a sub-domain with steep bed gradients.
[CFG] mode=smoke  tag=wind_on_diff_on_smoke  fp=b3e3d07331e9
[CFG] dt=5.0s  CFL limit=31.7s  ratio=0.158
[CFG] reference run: 37.3 h in 26,827 steps
[CFG] effective physics passes = 100.0  (V17 was 16.7)
[CFG] PDE coefficients: a1=16.1  a2=32.2  c_adv_x=0.268  c_adv_y=0.537  c_grad_x=29.2  c_grad_y=58.5  c_cor=2.91  c_fric=224  c_wind=2.42  c_diff_x=1.79e-05  c_diff_y=7.15e-05  D_floor=0.05
[CFG] saved -> /kaggle/working/v18/config_wind_on_diff_on_smoke.json


In [4]:
# =============================================================================
# CELL 2 / 9  --  REFERENCE ("GROUND TRUTH") SOLVER
# =============================================================================
# Arakawa C-grid, explicit forward Euler, first-order upwind momentum advection.
#
# Why a C-grid and not the V17 collocated A-grid:
#   1. The A-grid supports a pressure-velocity checkerboard mode. V17 had to
#      smooth the true Cf field to keep the solver stable (Validity Boundary
#      16), which capped how sharp a rock/mud transition the twin experiment
#      could even pose. On a C-grid that mode is absent.
#   2. Anti-inverse-crime. The PINN is collocated and meshless; the reference
#      is staggered and discrete. They now share only the continuum equations,
#      which is the strongest form of the claim the proposal wants to make
#      (Validity Boundary 15).
#
# ONE GROUND TRUTH, GENERATED WITH WIND ON, SHARED BY EVERY ABLATION ARM.
# In V17 the epoch-50 loss differed by a factor of 158 between the wind-on and
# wind-off arms, which means the truth itself was regenerated per arm. That
# makes H1b untestable: a Forcing-Aliasing test requires wind in the TRUTH and
# the wind term missing from the INVERSE MODEL, not two self-consistent worlds.
# =============================================================================

import numpy as np


# -----------------------------------------------------------------------------
# Static fields. Shared verbatim with the PINN (Cell 3) -- no separate copy.
# -----------------------------------------------------------------------------
def make_grid(cfg):
    d = cfg.domain
    xc = (np.arange(d.nx) + 0.5) * d.dx          # cell centres
    yc = (np.arange(d.ny) + 0.5) * d.dy
    return xc, yc


def bed_elevation(cfg):
    """b(x,y) <= 0, bed elevation. Returns b on cell centres, shape (nx, ny)."""
    d, p = cfg.domain, cfg.phys
    xc, yc = make_grid(cfg)
    X, Y = np.meshgrid(xc, yc, indexing="ij")

    yn = (Y / d.Ly) * 2.0 - 1.0                  # -1 .. +1 across strait
    xn = (X / d.Lx) * 2.0 - 1.0

    # Parabolic cross-section: deep mid-channel, shallow flanks
    depth = p.depth_min + (p.depth_max - p.depth_min) * (1.0 - yn ** 2) ** 1.2
    # Two rocky sills across the channel
    for x0, w in ((-0.35, 0.10), (0.30, 0.08)):
        depth -= p.ridge_amp * np.exp(-((xn - x0) / w) ** 2)
    depth = np.clip(depth, p.depth_min * 0.6, p.depth_max)
    return -depth


def true_cf(cfg):
    """Ground-truth Cf(x,y) on cell centres. Two smooth zones."""
    d, p = cfg.domain, cfg.phys
    xc, yc = make_grid(cfg)
    X, Y = np.meshgrid(xc, yc, indexing="ij")
    xn = (X / d.Lx) * 2.0 - 1.0
    yn = (Y / d.Ly) * 2.0 - 1.0

    w = p.cf_transition_cells * d.dx / (0.5 * d.Lx)   # transition in xn units
    # Rocky mid-channel band between the two sills, plus the sills themselves
    s = np.tanh((xn + 0.55) / w) * np.tanh((0.55 - xn) / w)
    s = 0.5 * (s + 1.0)
    s *= np.exp(-(yn / 0.75) ** 4)                    # rock only mid-channel
    cf = p.cf_mud + (p.cf_rock - p.cf_mud) * s
    return np.clip(cf, p.cf_lo * 1.05, p.cf_hi * 0.95)


def eddy_viscosity(cfg):
    """nu(x,y) and its gradients from the bed-slope proxy, on cell centres.

    Returned even when diffusion is disabled, so that Cell 8 can report the
    term's magnitude either way.
    """
    d, p = cfg.domain, cfg.phys
    b = bed_elevation(cfg)
    bx = np.gradient(b, d.dx, axis=0)
    by = np.gradient(b, d.dy, axis=1)
    slope = np.sqrt(bx ** 2 + by ** 2)
    p90 = np.percentile(slope, 90)
    s = 1.0 / (1.0 + np.exp(-p.nu_sigmoid_k * (slope / max(p90, 1e-12) - 1.0)))
    nu = p.nu_base + (p.nu_max - p.nu_base) * s
    nux = np.gradient(nu, d.dx, axis=0)
    nuy = np.gradient(nu, d.dy, axis=1)
    return nu, nux, nuy, slope, p90


# -----------------------------------------------------------------------------
# Forcing. Closed form, so the reference solver and the PINN evaluate the
# IDENTICAL function -- no interpolation error between them.
# -----------------------------------------------------------------------------
def tidal_elevation(t, cfg):
    p, r = cfg.phys, cfg.ref
    ramp = 1.0 - np.exp(-t / (r.ramp_cycles * p.T_m2))
    return ramp * (p.a_m2 * np.sin(p.omega_m2 * t)
                   + p.a_s2 * np.sin(p.omega_s2 * t + p.phase_s2))


def wind_stress(t, X, Y, cfg, xp=np):
    """Large & Pond (1981) surface stress, N/m^2. Vectorised over X, Y, t.

    Returns (tau_x, tau_y). Zero when cfg.wind_enabled is False.
    """
    if not cfg.wind_enabled:
        z = xp.zeros_like(xp.asarray(X, dtype=float))
        return z, z

    d, p, r = cfg.domain, cfg.phys, cfg.ref
    T_total = r.n_tidal_cycles * p.T_m2

    yn = (Y / d.Ly) * 2.0 - 1.0
    spatial = 1.0 + p.wind_spatial_amp * xp.cos(0.5 * xp.pi * yn)
    seasonal = 1.0 + p.wind_seasonal_amp * xp.sin(2.0 * xp.pi * t / T_total)
    W = p.wind_speed * spatial * seasonal

    th = xp.deg2rad(p.wind_dir_deg)
    ex, ey = -xp.sin(th), -xp.cos(th)             # unit vector wind blows TO

    # Smooth Large & Pond drag: 1.2e-3 below 11 m/s, (0.49+0.065W)e-3 above
    cd_lo = 1.2e-3
    cd_hi = (0.49 + 0.065 * W) * 1e-3
    blend = 1.0 / (1.0 + xp.exp(-(W - 11.0) / 0.5))
    cd = cd_lo * (1.0 - blend) + cd_hi * blend

    mag = p.rho_a * cd * W * W
    return mag * ex, mag * ey


# -----------------------------------------------------------------------------
# C-grid helpers
# -----------------------------------------------------------------------------
def _c2u(a):
    """Cell centres -> x-faces, shape (nx,ny) -> (nx+1,ny), edge-replicated."""
    inner = 0.5 * (a[:-1] + a[1:])
    return np.concatenate([a[:1], inner, a[-1:]], axis=0)


def _c2v(a):
    inner = 0.5 * (a[:, :-1] + a[:, 1:])
    return np.concatenate([a[:, :1], inner, a[:, -1:]], axis=1)


def _u2c(u):   return 0.5 * (u[:-1] + u[1:])
def _v2c(v):   return 0.5 * (v[:, :-1] + v[:, 1:])


def _v_at_u(v):
    """v on y-faces (nx,ny+1) -> v on x-faces (nx+1,ny)."""
    vc = _v2c(v)                        # (nx, ny)
    return _c2u(vc)


def _u_at_v(u):
    uc = _u2c(u)                        # (nx, ny)
    return _c2v(uc)


def _upwind_dadx(a, vel, dx):
    """First-order upwind d a/dx with the sign of vel, a and vel co-located."""
    fwd = np.empty_like(a); bwd = np.empty_like(a)
    fwd[:-1] = (a[1:] - a[:-1]) / dx; fwd[-1] = 0.0
    bwd[1:] = (a[1:] - a[:-1]) / dx;  bwd[0] = 0.0
    return np.where(vel > 0.0, bwd, fwd)


def _upwind_dady(a, vel, dy):
    fwd = np.empty_like(a); bwd = np.empty_like(a)
    fwd[:, :-1] = (a[:, 1:] - a[:, :-1]) / dy; fwd[:, -1] = 0.0
    bwd[:, 1:] = (a[:, 1:] - a[:, :-1]) / dy;  bwd[:, 0] = 0.0
    return np.where(vel > 0.0, bwd, fwd)


def _lap(a, dx, dy):
    out = np.zeros_like(a)
    out[1:-1, :] += (a[2:, :] - 2 * a[1:-1, :] + a[:-2, :]) / dx ** 2
    out[:, 1:-1] += (a[:, 2:] - 2 * a[:, 1:-1] + a[:, :-2]) / dy ** 2
    return out


# -----------------------------------------------------------------------------
# Main integrator
# -----------------------------------------------------------------------------
def run_reference(cfg, verbose=True):
    d, p, r = cfg.domain, cfg.phys, cfg.ref
    dx, dy, dt = d.dx, d.dy, r.dt

    b = bed_elevation(cfg)                  # (nx, ny), <= 0
    H = -b                                  # still-water depth
    cf = true_cf(cfg)
    nu, _, _, _, _ = eddy_viscosity(cfg)
    # The reference solver ALWAYS carries the full spatially varying viscosity.
    # The ablation switch only controls whether the INVERSE MODEL is told about
    # it. There is one truth, not one truth per ablation arm.
    nu_ref = p.nu_fd + nu

    xc, yc = make_grid(cfg)
    Xc, Yc = np.meshgrid(xc, yc, indexing="ij")
    Xu, Yu = np.meshgrid(np.arange(d.nx + 1) * dx, yc, indexing="ij")
    Xv, Yv = np.meshgrid(xc, np.arange(d.ny + 1) * dy, indexing="ij")

    cf_u, cf_v = _c2u(cf), _c2v(cf)
    nu_u, nu_v = _c2u(nu_ref), _c2v(nu_ref)

    eta = np.zeros((d.nx, d.ny))
    u = np.zeros((d.nx + 1, d.ny))
    v = np.zeros((d.nx, d.ny + 1))

    n_steps = int(r.n_tidal_cycles * p.T_m2 / dt)
    n_warm = int(r.warmup_cycles * p.T_m2 / dt)
    save_idx = np.unique(np.linspace(n_warm, n_steps, r.n_snapshots).astype(int))

    snaps = {k: [] for k in ("t", "eta", "u", "v")}
    clip_events = 0
    dmin_seen = np.inf
    umax_seen = 0.0

    for n in range(n_steps + 1):
        t = n * dt
        D = H + eta
        dmin_seen = min(dmin_seen, D.min())
        if D.min() < 0.5:
            clip_events += 1
            D = np.maximum(D, 0.5)

        Du, Dv = _c2u(D), _c2v(D)
        tau_xu, _ = wind_stress(t, Xu, Yu, cfg, np)
        _, tau_yv = wind_stress(t, Xv, Yv, cfg, np)

        # ---- u momentum on x-faces ---------------------------------------
        vu = _v_at_u(v)
        spd_u = np.sqrt(u ** 2 + vu ** 2)
        deta_dx = np.zeros_like(u)
        deta_dx[1:-1] = (eta[1:] - eta[:-1]) / dx

        du = (-u * _upwind_dadx(u, u, dx)
              - vu * _upwind_dady(u, vu, dy)
              - p.g * deta_dx
              + p.f_coriolis * vu
              - cf_u * spd_u * u / Du
              + tau_xu / (p.rho_w * Du)
              + nu_u * _lap(u, dx, dy))

        # ---- v momentum on y-faces ---------------------------------------
        uv = _u_at_v(u)
        spd_v = np.sqrt(uv ** 2 + v ** 2)
        deta_dy = np.zeros_like(v)
        deta_dy[:, 1:-1] = (eta[:, 1:] - eta[:, :-1]) / dy

        dv = (-uv * _upwind_dadx(v, uv, dx)
              - v * _upwind_dady(v, v, dy)
              - p.g * deta_dy
              - p.f_coriolis * uv
              - cf_v * spd_v * v / Dv
              + tau_yv / (p.rho_w * Dv)
              + nu_v * _lap(v, dx, dy))

        u = u + dt * du
        v = v + dt * dv

        # rigid walls: west face, both lateral faces
        u[0, :] = 0.0
        v[:, 0] = 0.0
        v[:, -1] = 0.0
        # eastern open boundary: zero-gradient normal velocity
        u[-1, :] = u[-2, :]

        # ---- continuity on cell centres ----------------------------------
        fx = Du * u                                  # (nx+1, ny)
        fy = Dv * v                                  # (nx, ny+1)
        eta = eta - dt * ((fx[1:] - fx[:-1]) / dx + (fy[:, 1:] - fy[:, :-1]) / dy)

        # prescribed tide in the easternmost column (Gulf of Oman)
        eta[-1, :] = tidal_elevation(t + dt, cfg)

        umax_seen = max(umax_seen, np.abs(u).max(), np.abs(v).max())
        if not np.isfinite(eta).all():
            raise FloatingPointError(f"reference solver diverged at step {n}")

        if n in save_idx:
            snaps["t"].append(t)
            snaps["eta"].append(eta.copy())
            snaps["u"].append(_u2c(u))               # interpolate to centres
            snaps["v"].append(_v2c(v))

        if verbose and n % max(1, n_steps // 10) == 0:
            print(f"  step {n:>7}/{n_steps}  t={t/3600:6.2f} h  "
                  f"|eta|max={np.abs(eta).max():.3f} m  "
                  f"|u|max={np.abs(u).max():.3f} m/s")

    ref = dict(
        t=np.asarray(snaps["t"]),
        eta=np.stack(snaps["eta"]),                  # (ns, nx, ny)
        u=np.stack(snaps["u"]),
        v=np.stack(snaps["v"]),
        b=b, H=H, cf_true=cf, nu=nu, nu_ref=nu_ref,
        xc=xc, yc=yc,
        clip_events=clip_events,
        min_depth_seen=float(dmin_seen),
        n_steps=n_steps,
    )
    if verbose:
        report_reference_scale(ref, cfg)
    return ref


# -----------------------------------------------------------------------------
# Validity Boundary 25: the scale audit, run BEFORE any metric is interpreted
# -----------------------------------------------------------------------------
def report_reference_scale(ref, cfg, strict=True):
    u, v, eta = ref["u"], ref["v"], ref["eta"]
    spd = np.sqrt(u ** 2 + v ** 2)
    rep = dict(
        clip_events=ref["clip_events"],
        min_depth_seen=ref["min_depth_seen"],
        u_min=float(u.min()), u_max=float(u.max()),
        v_min=float(v.min()), v_max=float(v.max()),
        u_range=float(u.max() - u.min()),
        v_range=float(v.max() - v.min()),
        speed_p99=float(np.percentile(spd, 99)),
        speed_mean=float(spd.mean()),
        eta_range=float(eta.max() - eta.min()),
    )
    print("\n[REF-SCALE] --- Validity Boundary 25 audit ---")
    print(f"  clip events                     : {rep['clip_events']}")
    print(f"  min water depth seen            : {rep['min_depth_seen']:.2f} m")
    print(f"  u  min / max / range            : {rep['u_min']:+.3f} / "
          f"{rep['u_max']:+.3f} / {rep['u_range']:.3f} m/s")
    print(f"  v  min / max / range            : {rep['v_min']:+.3f} / "
          f"{rep['v_max']:+.3f} / {rep['v_range']:.3f} m/s")
    print(f"  mean speed  (Johns et al. ~0.15): {rep['speed_mean']:.3f} m/s")
    print(f"  99th pct speed                  : {rep['speed_p99']:.3f} m/s")
    print(f"  eta range                       : {rep['eta_range']:.3f} m")

    assert rep["clip_events"] == 0, (
        f"{rep['clip_events']} depth-clip events: the reference solver is not "
        "clean. Do not build a twin experiment on it.")

    if strict:
        assert 0.03 <= rep["speed_mean"] <= 0.60, (
            f"mean speed {rep['speed_mean']:.3f} m/s is not a plausible Hormuz "
            "regime (observed layer exchange ~0.15 m/s, Johns et al. 2003). "
            "Retune Physics.a_m2 / wind_speed BEFORE interpreting any H1a "
            "percentage. This is the V17 failure mode.")
        assert rep["speed_p99"] <= 2.0, (
            f"99th percentile speed {rep['speed_p99']:.2f} m/s is too energetic. "
            "V17 ran at an implied ~11 m/s range, which INFLATES the H1a "
            "denominator and simultaneously makes Cf artificially easy to see "
            "(the friction term scales as |u|u).")
    print("[REF-SCALE] PASS\n")
    return rep


# -----------------------------------------------------------------------------
if __name__ == "__main__" or True:
    print("[CELL 2] generating reference fields ...")
    REF = run_reference(CFG)                                   # noqa: F821
    REF_SCALE = report_reference_scale(REF, CFG, strict=True)   # noqa: F821

[CELL 2] generating reference fields ...
  step       0/26827  t=  0.00 h  |eta|max=0.000 m  |u|max=0.000 m/s
  step    2682/26827  t=  3.73 h  |eta|max=0.690 m  |u|max=0.077 m/s
  step    5364/26827  t=  7.45 h  |eta|max=0.598 m  |u|max=0.391 m/s
  step    8046/26827  t= 11.18 h  |eta|max=0.497 m  |u|max=0.298 m/s
  step   10728/26827  t= 14.90 h  |eta|max=1.055 m  |u|max=0.217 m/s
  step   13410/26827  t= 18.62 h  |eta|max=0.389 m  |u|max=0.549 m/s
  step   16092/26827  t= 22.35 h  |eta|max=1.126 m  |u|max=0.311 m/s
  step   18774/26827  t= 26.07 h  |eta|max=0.900 m  |u|max=0.189 m/s
  step   21456/26827  t= 29.80 h  |eta|max=0.462 m  |u|max=0.397 m/s
  step   24138/26827  t= 33.52 h  |eta|max=1.131 m  |u|max=0.351 m/s
  step   26820/26827  t= 37.25 h  |eta|max=0.394 m  |u|max=0.231 m/s

[REF-SCALE] --- Validity Boundary 25 audit ---
  clip events                     : 0
  min water depth seen            : 13.82 m
  u  min / max / range            : -0.300 / +0.559 / 0.859 m/s
  v  m

In [5]:
# =============================================================================
# CELL 3 / 9  --  NON-DIMENSIONALISATION, SAMPLING POOLS, STATIC FIELDS
# =============================================================================
# Three jobs:
#   1. Map the reference fields into the normalised coordinates the network
#      sees, and VERIFY the derived PDE coefficients by plugging the TRUTH into
#      the non-dimensional residual. If the truth does not nearly satisfy the
#      residual, the coefficients are wrong and nothing downstream is
#      meaningful. V17 had no such check.
#   2. Build two genuinely independent point pools (observation vs collocation).
#   3. Expose b, db/dx, db/dy, nu, dnu/dx, dnu/dy as differentiable
#      interpolants. Gradients of the STATIC fields are precomputed on the grid
#      and interpolated, never differentiated through the interpolator -- that
#      is what the proposal specifies and it keeps them smooth.
# =============================================================================

import numpy as np
import jax
import jax.numpy as jnp


# -----------------------------------------------------------------------------
# Coordinate maps
# -----------------------------------------------------------------------------
def coord_maps(cfg, ref):
    d, p, r = cfg.domain, cfg.phys, cfg.ref
    t0 = r.warmup_cycles * p.T_m2
    t1 = r.n_tidal_cycles * p.T_m2
    return dict(t0=t0, t1=t1,
                to_xn=lambda x: 2.0 * x / d.Lx - 1.0,
                to_yn=lambda y: 2.0 * y / d.Ly - 1.0,
                to_tn=lambda t: 2.0 * (t - t0) / (t1 - t0) - 1.0,
                from_xn=lambda xn: 0.5 * (xn + 1.0) * d.Lx,
                from_yn=lambda yn: 0.5 * (yn + 1.0) * d.Ly,
                from_tn=lambda tn: t0 + 0.5 * (tn + 1.0) * (t1 - t0))


# -----------------------------------------------------------------------------
# Static fields in normalised units
# -----------------------------------------------------------------------------
def build_static_fields(cfg, ref):
    d, s = cfg.domain, cfg.scl
    Lxe, Lye = cfg.Lx_eff, cfg.Ly_eff

    b = ref["b"]                                     # (nx, ny), metres, <= 0
    b_star = b / s.H_D
    dbdx = np.gradient(b, d.dx, axis=0) * Lxe / s.H_D   # d b* / d x*
    dbdy = np.gradient(b, d.dy, axis=1) * Lye / s.H_D

    nu = ref["nu"]                                   # m^2/s
    dnudx = np.gradient(nu, d.dx, axis=0) * Lxe
    dnudy = np.gradient(nu, d.dy, axis=1) * Lye

    if cfg.diffusion_enabled:
        assert nu.std() > 1e-6, (
            "nu(x,y) is spatially constant: the diffusion ablation would be "
            "inert by construction (the V17 result). Check nu_max / "
            "nu_sigmoid_k / the bed-slope field.")

    fields = {k: jnp.asarray(vv) for k, vv in dict(
        b=b_star, dbdx=dbdx, dbdy=dbdy,
        nu=nu, dnudx=dnudx, dnudy=dnudy,
        cf_true=ref["cf_true"]).items()}
    fields["nu_std"] = float(nu.std())
    fields["nu_min"] = float(nu.min())
    fields["nu_max"] = float(nu.max())
    return fields


def bilinear(field, xn, yn, nx, ny):
    """Bilinear sample of a cell-centred field at normalised (xn, yn) in [-1,1].

    Cell centres sit at xn = (2i+1)/nx - 1, so the continuous index is
    i = (xn+1)*nx/2 - 0.5.
    """
    fi = (xn + 1.0) * nx * 0.5 - 0.5
    fj = (yn + 1.0) * ny * 0.5 - 0.5
    fi = jnp.clip(fi, 0.0, nx - 1.0)
    fj = jnp.clip(fj, 0.0, ny - 1.0)
    i0 = jnp.floor(fi).astype(jnp.int32); i1 = jnp.minimum(i0 + 1, nx - 1)
    j0 = jnp.floor(fj).astype(jnp.int32); j1 = jnp.minimum(j0 + 1, ny - 1)
    wi = fi - i0; wj = fj - j0
    f00 = field[i0, j0]; f10 = field[i1, j0]
    f01 = field[i0, j1]; f11 = field[i1, j1]
    return ((1 - wi) * (1 - wj) * f00 + wi * (1 - wj) * f10
            + (1 - wi) * wj * f01 + wi * wj * f11)


def make_static_sampler(cfg, fields):
    nx, ny = cfg.domain.nx, cfg.domain.ny
    keys = ("b", "dbdx", "dbdy", "nu", "dnudx", "dnudy")
    packed = jnp.stack([fields[k] for k in keys])        # (6, nx, ny)

    def sample(xn, yn):
        vals = jax.vmap(lambda f: bilinear(f, xn, yn, nx, ny))(packed)
        return {k: vals[i] for i, k in enumerate(keys)}
    return sample


# -----------------------------------------------------------------------------
# Wind stress in normalised coordinates (closed form, shared with Cell 2)
# -----------------------------------------------------------------------------
def make_wind_sampler(cfg, maps):
    d = cfg.domain
    t0, t1 = maps["t0"], maps["t1"]

    def sample(tn, xn, yn):
        t = t0 + 0.5 * (tn + 1.0) * (t1 - t0)
        X = 0.5 * (xn + 1.0) * d.Lx
        Y = 0.5 * (yn + 1.0) * d.Ly
        return wind_stress(t, X, Y, cfg, jnp)          # noqa: F821  (Cell 2)
    return sample


# -----------------------------------------------------------------------------
# Observation pool: a deliberately crude satellite-track proxy
# -----------------------------------------------------------------------------
def build_observation_pool(cfg, ref, maps, rng):
    d, p, o, s = cfg.domain, cfg.phys, cfg.obs, cfg.scl
    ns = ref["t"].size
    xc, yc = ref["xc"], ref["yc"]

    ang = np.deg2rad(o.track_angle_deg)
    x_off = np.linspace(-d.Ly / np.tan(ang) - d.Lx * 0.05,
                        d.Lx * 1.05, o.n_obs_tracks
                        if hasattr(o, "n_obs_tracks") else cfg.cost.n_obs_tracks)

    pts = []
    for k, x0 in enumerate(x_off):
        L = d.Ly / np.sin(ang)
        n_al = max(2, int(L / o.along_track_spacing_m))
        ss = np.linspace(0.0, L, n_al)
        xs = x0 + ss * np.cos(ang)
        ys = ss * np.sin(ang)
        keep = (xs >= 0) & (xs <= d.Lx) & (ys >= 0) & (ys <= d.Ly)
        xs, ys = xs[keep], ys[keep]
        # each track is revisited only every `revisit_snapshots` snapshots,
        # with a per-track offset -> genuinely asynchronous multi-rate sampling
        for it in range(k % o.revisit_snapshots, ns, o.revisit_snapshots):
            for xx, yy in zip(xs, ys):
                pts.append((it, xx, yy))
    pts = np.asarray(pts, dtype=float)
    it = pts[:, 0].astype(int); xs = pts[:, 1]; ys = pts[:, 2]

    # bilinear read-out of the truth at the track locations
    def rd(arr):
        fi = xs / d.dx - 0.5; fj = ys / d.dy - 0.5
        i0 = np.clip(np.floor(fi).astype(int), 0, d.nx - 1)
        j0 = np.clip(np.floor(fj).astype(int), 0, d.ny - 1)
        i1 = np.minimum(i0 + 1, d.nx - 1); j1 = np.minimum(j0 + 1, d.ny - 1)
        wi = np.clip(fi - i0, 0, 1); wj = np.clip(fj - j0, 0, 1)
        return ((1 - wi) * (1 - wj) * arr[it, i0, j0]
                + wi * (1 - wj) * arr[it, i1, j0]
                + (1 - wi) * wj * arr[it, i0, j1]
                + wi * wj * arr[it, i1, j1])

    h_obs = rd(ref["eta"]) / s.H_eta
    noise = o.noise_pct / 100.0
    h_obs = h_obs + rng.normal(0.0, noise * np.abs(h_obs).std(), h_obs.shape)

    pool = dict(
        tn=maps["to_tn"](ref["t"][it]),
        xn=maps["to_xn"](xs),
        yn=maps["to_yn"](ys),
        h=h_obs,
        has_uv=bool(o.observe_velocity),
    )
    if o.observe_velocity:
        u_obs = rd(ref["u"]) / s.U
        v_obs = rd(ref["v"]) / s.U
        pool["u"] = u_obs + rng.normal(0, noise * np.abs(u_obs).std(), u_obs.shape)
        pool["v"] = v_obs + rng.normal(0, noise * np.abs(v_obs).std(), v_obs.shape)
    else:
        pool["u"] = np.zeros_like(h_obs)
        pool["v"] = np.zeros_like(h_obs)

    pool = {k: (jnp.asarray(v) if isinstance(v, np.ndarray) else v)
            for k, v in pool.items()}
    pool["n"] = int(h_obs.size)
    return pool


# -----------------------------------------------------------------------------
# Collocation pool: sampled independently of the observations
# -----------------------------------------------------------------------------
def build_collocation_pool(cfg, rng):
    n = cfg.cost.pool_phys
    # Stratified in time so no tidal phase is under-represented, uniform in
    # space. Sobol would be better but adds a scipy dependency at no real gain.
    tn = (rng.permutation(n) + rng.random(n)) / n * 2.0 - 1.0
    xn = rng.random(n) * 2.0 - 1.0
    yn = rng.random(n) * 2.0 - 1.0
    return dict(tn=jnp.asarray(tn), xn=jnp.asarray(xn), yn=jnp.asarray(yn), n=n)


def build_boundary_pool(cfg, rng, n=None):
    """Rigid-wall points: west face and both lateral faces (V_n = 0)."""
    n = n or max(2000, cfg.cost.batch_phys // 4)
    tn = rng.random(n) * 2.0 - 1.0
    which = rng.integers(0, 3, n)
    xn = np.where(which == 0, -1.0, rng.random(n) * 2.0 - 1.0)
    yn = np.where(which == 0, rng.random(n) * 2.0 - 1.0,
                  np.where(which == 1, -1.0, 1.0))
    normal = np.where(which == 0, 0, 1)     # 0 -> u=0, 1 -> v=0
    return dict(tn=jnp.asarray(tn), xn=jnp.asarray(xn), yn=jnp.asarray(yn),
                normal=jnp.asarray(normal), n=n)


# -----------------------------------------------------------------------------
# Evaluation grid (dense, full trajectory, post-warmup only)
# -----------------------------------------------------------------------------
def build_eval_set(cfg, ref, maps):
    d, s = cfg.domain, cfg.scl
    ns = ref["t"].size
    tn = maps["to_tn"](ref["t"])
    xn = maps["to_xn"](ref["xc"])
    yn = maps["to_yn"](ref["yc"])
    TN, XN, YN = np.meshgrid(tn, xn, yn, indexing="ij")
    return dict(
        tn=jnp.asarray(TN.ravel()), xn=jnp.asarray(XN.ravel()),
        yn=jnp.asarray(YN.ravel()),
        shape=(ns, d.nx, d.ny),
        h=jnp.asarray(ref["eta"].ravel() / s.H_eta),
        u=jnp.asarray(ref["u"].ravel() / s.U),
        v=jnp.asarray(ref["v"].ravel() / s.U),
        cf_true=jnp.asarray(ref["cf_true"]),
        xn_grid=jnp.asarray(xn), yn_grid=jnp.asarray(yn),
        n_snapshots=ns,
    )


# -----------------------------------------------------------------------------
# THE GUARD: does the ground truth satisfy the non-dimensional residual?
# -----------------------------------------------------------------------------
def verify_pde_coeffs(cfg, ref, tol_rel=0.50, strict=True, verbose=True):
    """Plug the reference solution into the non-dimensional SWE residual using
    centred finite differences on the saved snapshots.

    The residual will not be zero -- snapshot spacing is coarse in time and the
    reference used upwind advection -- but each term's magnitude must be
    physically sensible and the residual must be small COMPARED TO the leading
    terms. If it is not, a coefficient or a scale is wrong.

    This single check catches: a factor-of-2 half-span error, a swapped Lx/Ly,
    a wrong H_D, a sign error on Coriolis, and a wind-stress unit error.
    """
    d, p, s = cfg.domain, cfg.phys, cfg.scl
    co = cfg.pde_coeffs()
    dtn = 2.0 / (ref["t"].size - 1)
    dxn = 2.0 / d.nx
    dyn = 2.0 / d.ny

    h = ref["eta"] / s.H_eta
    u = ref["u"] / s.U
    v = ref["v"] / s.U
    eps_h = s.H_eta / s.H_D
    b = ref["b"] / s.H_D
    D = np.maximum(eps_h * h - b[None], co["D_floor"])

    sl = (slice(1, -1), slice(1, -1), slice(1, -1))
    ht = (h[2:] - h[:-2]) / (2 * dtn)
    hx = (h[:, 2:] - h[:, :-2]) / (2 * dxn)
    hy = (h[:, :, 2:] - h[:, :, :-2]) / (2 * dyn)
    ut = (u[2:] - u[:-2]) / (2 * dtn)
    ux = (u[:, 2:] - u[:, :-2]) / (2 * dxn)
    uy = (u[:, :, 2:] - u[:, :, :-2]) / (2 * dyn)

    Du = D * u
    Dv = D * v
    Dux = (Du[:, 2:] - Du[:, :-2]) / (2 * dxn)
    Dvy = (Dv[:, :, 2:] - Dv[:, :, :-2]) / (2 * dyn)

    R_mass = (ht[:, 1:-1, 1:-1]
              + co["a1"] * Dux[1:-1, :, 1:-1]
              + co["a2"] * Dvy[1:-1, 1:-1, :])

    xc, yc = ref["xc"], ref["yc"]
    T3, X3, Y3 = np.meshgrid(ref["t"], xc, yc, indexing="ij")
    tau_x, _ = wind_stress(T3, X3, Y3, cfg, np)        # noqa: F821
    spd = np.sqrt(u ** 2 + v ** 2)
    cfT = ref["cf_true"][None]

    T_ut = ut[:, 1:-1, 1:-1]
    T_adv = (co["c_adv_x"] * u[1:-1, 1:-1, 1:-1] * ux[1:-1, :, 1:-1]
             + co["c_adv_y"] * v[1:-1, 1:-1, 1:-1] * uy[1:-1, 1:-1, :])
    T_grad = co["c_grad_x"] * hx[1:-1, :, 1:-1]
    T_cor = -co["c_cor"] * v[1:-1, 1:-1, 1:-1]
    T_fric = (co["c_fric"] * cfT * spd * u / D)[1:-1, 1:-1, 1:-1]
    T_wind = -(co["c_wind"] * tau_x / D)[1:-1, 1:-1, 1:-1]
    R_u = T_ut + T_adv + T_grad + T_cor + T_fric + T_wind

    def rms(a): return float(np.sqrt(np.mean(np.asarray(a) ** 2)))
    mags = dict(u_t=rms(T_ut), advection=rms(T_adv), pressure_grad=rms(T_grad),
                coriolis=rms(T_cor), friction=rms(T_fric), wind=rms(T_wind))
    lead = max(mags.values())
    rel_u = rms(R_u) / lead
    rel_m = rms(R_mass) / max(rms(co["a1"] * Dux[1:-1, :, 1:-1]), 1e-30)

    if verbose:
        print("\n[PDE-CHECK] RMS of each non-dimensional u-momentum term:")
        for k, vv in sorted(mags.items(), key=lambda kv: -kv[1]):
            print(f"    {k:<16} {vv:10.4f}   ({vv/lead*100:5.1f}% of leading)")
        print(f"  residual R_u   / leading term : {rel_u:.3f}")
        print(f"  residual R_mass/ leading term : {rel_m:.3f}")
        print("  (non-zero is expected: snapshot time spacing is coarse and "
              "the reference used upwind advection)")

    # ---- A-PRIORI IDENTIFIABILITY ESTIMATE -------------------------------
    # The share of the momentum balance carried by the friction term is an
    # upper bound on how much of the observable signal Cf can possibly
    # explain. Compared against the observation noise, it says in advance
    # whether Cf is recoverable AT ALL -- before a single epoch is trained.
    fric_share = mags["friction"] / lead
    wind_share = mags["wind"] / lead
    noise = cfg.obs.noise_pct / 100.0
    snr = fric_share / max(noise, 1e-12)
    if verbose:
        print(f"\n[IDENTIFIABILITY] friction term = {fric_share*100:.1f}% of the "
              f"leading momentum term")
        print(f"[IDENTIFIABILITY] wind term     = {wind_share*100:.1f}%")
        print(f"[IDENTIFIABILITY] observation noise = {noise*100:.1f}%  "
              f"-> crude signal/noise = {snr:.1f}")
        if wind_share > fric_share:
            print("  NOTE: the wind term is LARGER than the friction term. An "
                  "inverse\n        model without a wind term must absorb that "
                  "forcing into Cf,\n        which is a first-principles "
                  "justification for H1b -- quantify it\n        in Section 1.2 "
                  "rather than asserting it.")
        if snr < 3.0:
            print("  WARNING: Cf carries only a few times the observation noise "
                  "in the\n           momentum balance. A 5% RMSE target on a "
                  "free continuous field\n           is very unlikely to be "
                  "reachable in this regime. Consider a\n           "
                  "low-dimensional Cf basis, or reduce the noise level, or add "
                  "an\n           observable with stronger friction sensitivity "
                  "(phase lag).")

    msg = None
    if rel_u >= tol_rel:
        msg = (f"ground truth does not satisfy the non-dimensional u-momentum "
               f"residual (rel={rel_u:.2f} > {tol_rel}). A scale or coefficient "
               "is probably wrong; fix this before training anything.")
    elif rel_m >= tol_rel:
        msg = (f"ground truth violates non-dimensional continuity "
               f"(rel={rel_m:.2f} > {tol_rel}).")
    if msg and strict:
        raise AssertionError(msg)
    if msg:
        print(f"[PDE-CHECK] WARNING (strict=False): {msg}")
    elif verbose:
        print("[PDE-CHECK] PASS\n")
    return dict(term_rms=mags, rel_residual_u=rel_u, rel_residual_mass=rel_m,
                friction_share_of_leading=float(fric_share),
                wind_share_of_leading=float(wind_share),
                friction_to_noise_ratio=float(snr))


# -----------------------------------------------------------------------------
def prepare_all(cfg, ref, seed=None):
    rng = np.random.default_rng(seed if seed is not None else cfg.seed)
    maps = coord_maps(cfg, ref)
    fields = build_static_fields(cfg, ref)
    data = dict(
        maps=maps,
        fields=fields,
        static=make_static_sampler(cfg, fields),
        wind=make_wind_sampler(cfg, maps),
        obs=build_observation_pool(cfg, ref, maps, rng),
        colloc=build_collocation_pool(cfg, rng),
        bc=build_boundary_pool(cfg, rng),
        eval=build_eval_set(cfg, ref, maps),
        eps_h=cfg.scl.H_eta / cfg.scl.H_D,
        coeffs=cfg.pde_coeffs(),
    )
    print(f"[DATA] observations : {data['obs']['n']:,} points "
          f"({'h,u,v' if data['obs']['has_uv'] else 'h only'})")
    print(f"[DATA] collocation  : {data['colloc']['n']:,} points")
    print(f"[DATA] boundary     : {data['bc']['n']:,} points")
    print(f"[DATA] nu range     : {fields['nu_min']:.1f} - "
          f"{fields['nu_max']:.1f} m^2/s (std {fields['nu_std']:.2f})")
    return data


if __name__ == "__main__" or True:
    PDE_CHECK = verify_pde_coeffs(CFG, REF)          # noqa: F821
    DATA = prepare_all(CFG, REF)                     # noqa: F821


[PDE-CHECK] RMS of each non-dimensional u-momentum term:
    u_t                  2.2380   (100.0% of leading)
    pressure_grad        2.2053   ( 98.5% of leading)
    wind                 0.2005   (  9.0% of leading)
    coriolis             0.1999   (  8.9% of leading)
    friction             0.1445   (  6.5% of leading)
    advection            0.0886   (  4.0% of leading)
  residual R_u   / leading term : 0.061
  residual R_mass/ leading term : 0.101
  (non-zero is expected: snapshot time spacing is coarse and the reference used upwind advection)

[IDENTIFIABILITY] friction term = 6.5% of the leading momentum term
[IDENTIFIABILITY] wind term     = 9.0%
[IDENTIFIABILITY] observation noise = 3.0%  -> crude signal/noise = 2.2
  NOTE: the wind term is LARGER than the friction term. An inverse
        model without a wind term must absorb that forcing into Cf,
        which is a first-principles justification for H1b -- quantify it
        in Section 1.2 rather than asserting it.
   

In [6]:
# =============================================================================
# CELL 4 / 9  --  DUAL-BRANCH STATE-AUGMENTED NETWORK
# =============================================================================
# Pure JAX, explicit parameter dicts. No Haiku / Flax dependency: one less
# Kaggle-image variable, and the parameter tree is trivially flattenable for
# scipy L-BFGS-B.
#
# Hard structural constraint: the friction branch takes only (x*, y*), so
# dCf/dt == 0 identically. It is enforced by the architecture, not the loss.
#
# THE KEY CHANGE FROM V17 -- friction output parameterisation.
#   V17: Cf = lo + (hi-lo) * sigmoid(z).
#        Measured outcome: sigmoid_deriv_mean fell from 0.243 to 0.018 against
#        a theoretical maximum of 0.25, i.e. ~93% saturated at the end of
#        training. The gradient reaching the friction branch was throttled by
#        an order of magnitude exactly when it mattered most. Validity
#        Boundary 6 was ACTIVE, not mitigated.
#   V18: Cf = exp(log(lo) + (log(hi)-log(lo)) * sigmoid(z / temp))
#        - log space gives uniform RELATIVE resolution across a band that
#          spans 5.3x, which matches how Cf actually matters (it enters as a
#          multiplicative coefficient);
#        - temp > 1 widens the usable pre-activation range;
#        - a weak L2 penalty on z (Arch.fric_saturation_penalty) actively pulls
#          the pre-activation back toward the responsive region;
#        - `sigmoid_deriv_mean` and `saturated_fraction` are returned as
#          telemetry on every logged step so this failure can never hide again.
# =============================================================================

import numpy as np
import jax
import jax.numpy as jnp


# -----------------------------------------------------------------------------
# Initialisation
# -----------------------------------------------------------------------------
def _mlp_init(key, sizes):
    params = []
    for i, (fan_in, fan_out) in enumerate(zip(sizes[:-1], sizes[1:])):
        key, sk = jax.random.split(key)
        # Xavier/Glorot, appropriate for tanh
        lim = np.sqrt(6.0 / (fan_in + fan_out))
        w = jax.random.uniform(sk, (fan_in, fan_out), minval=-lim, maxval=lim)
        params.append(dict(w=w, b=jnp.zeros((fan_out,))))
    return params


def init_params(cfg, key):
    a, p = cfg.arch, cfg.phys
    k1, k2, k3, k4 = jax.random.split(key, 4)

    # Fourier feature matrices are FIXED (not trained): Tancik et al. (2020)
    B_hydro = jax.random.normal(k1, (3, a.fourier_features)) * a.sigma_hydro
    B_fric = jax.random.normal(k2, (2, a.fourier_features)) * a.sigma_fric

    d_in = 2 * a.fourier_features
    hydro = _mlp_init(k3, (d_in,) + tuple(a.hydro_layers) + (3,))
    fric = _mlp_init(k4, (d_in,) + tuple(a.fric_layers) + (1,))

    # Analytic bias initialisation for the friction head: start at the
    # geometric mean of the band, i.e. sigmoid(z)=0.5, i.e. z=0. With
    # zero-initialised biases that is already true, but we set it explicitly so
    # that changing the parameterisation cannot silently change the prior.
    fric[-1]["b"] = jnp.zeros((1,))

    return dict(B_hydro=B_hydro, B_fric=B_fric, hydro=hydro, fric=fric)


def trainable_keys():
    """Which top-level entries the optimiser may touch. B matrices are frozen."""
    return ("hydro", "fric")


def split_params(params):
    """Separate the two branches so freeze-release can address them."""
    return dict(hydro=params["hydro"]), dict(fric=params["fric"])


# -----------------------------------------------------------------------------
# Forward passes
# -----------------------------------------------------------------------------
def _fourier(x, B):
    proj = 2.0 * jnp.pi * (x @ B)
    return jnp.concatenate([jnp.sin(proj), jnp.cos(proj)], axis=-1)


def _mlp(params, x):
    for layer in params[:-1]:
        x = jnp.tanh(x @ layer["w"] + layer["b"])
    last = params[-1]
    return x @ last["w"] + last["b"]


def hydro_forward(params, cfg, tn, xn, yn):
    """Returns (h*, u*, v*), unbounded. Inputs are 1-D arrays in [-1, 1]."""
    a = cfg.arch
    ys = yn * (cfg.domain.Lx / cfg.domain.Ly) if a.y_aspect_rescale else yn
    # Rescaling the across-strait input equalises the Fourier-feature variance
    # in both directions for a 2:1 domain, so sigma_hydro means the same thing
    # along and across the strait.
    x = jnp.stack([tn, xn, ys], axis=-1)
    out = _mlp(params["hydro"], _fourier(x, params["B_hydro"]))
    return out[..., 0], out[..., 1], out[..., 2]


def fric_pre_activation(params, cfg, xn, yn):
    a = cfg.arch
    ys = yn * (cfg.domain.Lx / cfg.domain.Ly) if a.y_aspect_rescale else yn
    x = jnp.stack([xn, ys], axis=-1)
    return _mlp(params["fric"], _fourier(x, params["B_fric"]))[..., 0]


def cf_from_pre_activation(z, cfg):
    a, p = cfg.arch, cfg.phys
    s = jax.nn.sigmoid(z / a.fric_sigmoid_temp)
    if a.fric_output == "log_sigmoid":
        lo, hi = jnp.log(p.cf_lo), jnp.log(p.cf_hi)
        return jnp.exp(lo + (hi - lo) * s)
    elif a.fric_output == "linear_sigmoid":     # the V17 behaviour, for A/B use
        return p.cf_lo + (p.cf_hi - p.cf_lo) * s
    raise ValueError(a.fric_output)


def fric_forward(params, cfg, xn, yn):
    z = fric_pre_activation(params, cfg, xn, yn)
    return cf_from_pre_activation(z, cfg), z


def forward_all(params, cfg, tn, xn, yn):
    h, u, v = hydro_forward(params, cfg, tn, xn, yn)
    cf, z = fric_forward(params, cfg, xn, yn)
    return dict(h=h, u=u, v=v, cf=cf, z=z)


# -----------------------------------------------------------------------------
# Saturation telemetry -- the diagnostic V17 logged but never acted on
# -----------------------------------------------------------------------------
def saturation_stats(params, cfg, xn, yn):
    z = fric_pre_activation(params, cfg, xn, yn)
    zt = z / cfg.arch.fric_sigmoid_temp
    s = jax.nn.sigmoid(zt)
    deriv = s * (1.0 - s)                       # max 0.25
    return dict(
        sigmoid_deriv_mean=jnp.mean(deriv),
        sigmoid_deriv_min=jnp.min(deriv),
        # fraction of points whose response is below 20% of the maximum
        saturated_fraction=jnp.mean((deriv < 0.05).astype(jnp.float64)),
        z_abs_mean=jnp.mean(jnp.abs(z)),
        z_abs_max=jnp.max(jnp.abs(z)),
        cf_mean=jnp.mean(cf_from_pre_activation(z, cfg)),
    )


# -----------------------------------------------------------------------------
# Parameter counting / memory profiling for H1c
# -----------------------------------------------------------------------------
def count_params(params):
    leaves = jax.tree_util.tree_leaves(params)
    return int(sum(np.prod(np.shape(l)) for l in leaves))


def vram_peak_mb():
    """Peak device bytes, or None on CPU. Reported honestly either way -- V17
    recorded peak_vram_mb=None in every run while H1c claimed a >60% VRAM
    reduction, i.e. the hypothesis had no measurement behind it at all."""
    try:
        dev = jax.devices()[0]
        if dev.platform == "cpu":
            return None
        st = dev.memory_stats()
        if not st:
            return None
        return float(st.get("peak_bytes_in_use", 0)) / 1024 ** 2
    except Exception:
        return None


if __name__ == "__main__" or True:
    _key = jax.random.PRNGKey(CFG.seed)                       # noqa: F821
    PARAMS0 = init_params(CFG, _key)                          # noqa: F821
    print(f"[MODEL] parameters: {count_params(PARAMS0):,}  "
          f"(hydro {count_params(PARAMS0['hydro']):,} / "
          f"fric {count_params(PARAMS0['fric']):,})")
    print(f"[MODEL] friction output: {CFG.arch.fric_output}, "                # noqa: F821
          f"temp={CFG.arch.fric_sigmoid_temp}, "                              # noqa: F821
          f"band [{CFG.phys.cf_lo}, {CFG.phys.cf_hi}]")                       # noqa: F821
    print(f"[MODEL] device: {jax.devices()[0].platform}, "
          f"x64={jax.config.jax_enable_x64}")

[MODEL] parameters: 116,420  (hydro 66,435 / fric 49,665)
[MODEL] friction output: log_sigmoid, temp=2.0, band [0.0015, 0.008]
[MODEL] device: gpu, x64=True


In [7]:
# =============================================================================
# CELL 5 / 9  --  GOVERNING EQUATIONS (forward-mode JVP)
# =============================================================================
# Derivative budget, exactly as the proposal specifies and no more:
#   * 1 single JVP  -> h_t, u_t, v_t
#   * 1 nested JVP in x -> h_x, u_x, v_x AND u_xx, v_xx   (primal + tangent)
#   * 1 nested JVP in y -> h_y, u_y, v_y AND u_yy, v_yy
# The first derivatives are read off the PRIMAL output of the nested call, so
# nothing is computed twice. A naive implementation issues four separate calls
# (component x direction) and doubles the cost -- which would directly corrupt
# the H1c number. `derivative_call_count()` below reports what was actually
# executed, so the H1c figure can be audited rather than asserted.
#
# When diffusion is disabled the nesting is skipped entirely (3 single JVPs),
# which is the honest baseline for the H1c comparison.
# =============================================================================

import jax
import jax.numpy as jnp


_CALL_LOG = {"single_jvp": 0, "nested_jvp": 0}


def derivative_call_count():
    return dict(_CALL_LOG)


# -----------------------------------------------------------------------------
def _state_fn(params, cfg):
    def f(tn, xn, yn):
        h, u, v = hydro_forward(params, cfg, tn, xn, yn)      # noqa: F821
        return (h, u, v)
    return f


def compute_derivatives(params, cfg, tn, xn, yn, need_second: bool):
    f = _state_fn(params, cfg)
    ones = jnp.ones_like(tn)
    zeros = jnp.zeros_like(tn)

    # ---- time -------------------------------------------------------------
    (h, u, v), (h_t, u_t, v_t) = jax.jvp(
        lambda t: f(t, xn, yn), (tn,), (ones,))
    _CALL_LOG["single_jvp"] += 1

    if not need_second:
        _, (h_x, u_x, v_x) = jax.jvp(lambda x: f(tn, x, yn), (xn,), (ones,))
        _, (h_y, u_y, v_y) = jax.jvp(lambda y: f(tn, xn, y), (yn,), (ones,))
        _CALL_LOG["single_jvp"] += 2
        z2 = zeros
        return dict(h=h, u=u, v=v, h_t=h_t, u_t=u_t, v_t=v_t,
                    h_x=h_x, u_x=u_x, v_x=v_x, h_y=h_y, u_y=u_y, v_y=v_y,
                    u_xx=z2, v_xx=z2, u_yy=z2, v_yy=z2)

    # ---- x: one nested call gives first AND second derivatives -------------
    def dfdx(x):
        _, tang = jax.jvp(lambda xx: f(tn, xx, yn), (x,), (ones,))
        return tang                                   # (h_x, u_x, v_x)

    (h_x, u_x, v_x), (_, u_xx, v_xx) = jax.jvp(dfdx, (xn,), (ones,))

    # ---- y ----------------------------------------------------------------
    def dfdy(y):
        _, tang = jax.jvp(lambda yy: f(tn, xn, yy), (y,), (ones,))
        return tang

    (h_y, u_y, v_y), (_, u_yy, v_yy) = jax.jvp(dfdy, (yn,), (ones,))
    _CALL_LOG["nested_jvp"] += 2

    return dict(h=h, u=u, v=v, h_t=h_t, u_t=u_t, v_t=v_t,
                h_x=h_x, u_x=u_x, v_x=v_x, h_y=h_y, u_y=u_y, v_y=v_y,
                u_xx=u_xx, v_xx=v_xx, u_yy=u_yy, v_yy=v_yy)


# -----------------------------------------------------------------------------
def swe_residuals(params, cfg, data, tn, xn, yn, return_terms=False):
    co = data["coeffs"]
    eps_h = data["eps_h"]
    need_second = bool(cfg.diffusion_enabled)

    d = compute_derivatives(params, cfg, tn, xn, yn, need_second)
    st = data["static"](xn, yn)
    cf, z = fric_forward(params, cfg, xn, yn)                 # noqa: F821
    tau_x, tau_y = data["wind"](tn, xn, yn)

    # ---- water depth, with a masked floor ---------------------------------
    D_raw = eps_h * d["h"] - st["b"]
    D = jnp.maximum(D_raw, co["D_floor"])
    live = (D_raw > co["D_floor"]).astype(D.dtype)
    D_x = live * (eps_h * d["h_x"] - st["dbdx"])
    D_y = live * (eps_h * d["h_y"] - st["dbdy"])

    spd = jnp.sqrt(d["u"] ** 2 + d["v"] ** 2 + 1e-12)

    # ---- continuity -------------------------------------------------------
    R_mass = (d["h_t"]
              + co["a1"] * (D * d["u_x"] + D_x * d["u"])
              + co["a2"] * (D * d["v_y"] + D_y * d["v"]))

    # ---- momentum ---------------------------------------------------------
    adv_u = co["c_adv_x"] * d["u"] * d["u_x"] + co["c_adv_y"] * d["v"] * d["u_y"]
    adv_v = co["c_adv_x"] * d["u"] * d["v_x"] + co["c_adv_y"] * d["v"] * d["v_y"]
    grad_u = co["c_grad_x"] * d["h_x"]
    grad_v = co["c_grad_y"] * d["h_y"]
    cor_u = -co["c_cor"] * d["v"]
    cor_v = +co["c_cor"] * d["u"]
    fric_u = co["c_fric"] * cf * spd * d["u"] / D
    fric_v = co["c_fric"] * cf * spd * d["v"] / D
    wind_u = -co["c_wind"] * tau_x / D
    wind_v = -co["c_wind"] * tau_y / D

    if cfg.diffusion_enabled:
        diff_u = (co["c_diff_x"] * (st["nu"] * d["u_xx"] + st["dnudx"] * d["u_x"])
                  + co["c_diff_y"] * (st["nu"] * d["u_yy"] + st["dnudy"] * d["u_y"]))
        diff_v = (co["c_diff_x"] * (st["nu"] * d["v_xx"] + st["dnudx"] * d["v_x"])
                  + co["c_diff_y"] * (st["nu"] * d["v_yy"] + st["dnudy"] * d["v_y"]))
    else:
        diff_u = jnp.zeros_like(d["u"])
        diff_v = jnp.zeros_like(d["u"])

    R_u = d["u_t"] + adv_u + grad_u + cor_u + fric_u + wind_u - diff_u
    R_v = d["v_t"] + adv_v + grad_v + cor_v + fric_v + wind_v - diff_v

    out = dict(R_mass=R_mass, R_u=R_u, R_v=R_v, cf=cf, z=z, D=D,
               h=d["h"], u=d["u"], v=d["v"])
    if return_terms:
        out["terms"] = dict(
            u_t=d["u_t"], advection=adv_u, pressure_grad=grad_u,
            coriolis=cor_u, friction=fric_u, wind=wind_u, diffusion=diff_u,
            mass_h_t=d["h_t"],
            mass_div_x=co["a1"] * (D * d["u_x"] + D_x * d["u"]),
            mass_div_y=co["a2"] * (D * d["v_y"] + D_y * d["v"]),
        )
    return out


# -----------------------------------------------------------------------------
def boundary_residual(params, cfg, data, bc):
    """Rigid wall: normal velocity vanishes. normal==0 -> u, normal==1 -> v."""
    h, u, v = hydro_forward(params, cfg, bc["tn"], bc["xn"], bc["yn"])  # noqa
    return jnp.where(bc["normal"] == 0, u, v)


# -----------------------------------------------------------------------------
# Term magnitude report. THE diagnostic that makes a term like adaptive
# diffusion impossible to mis-sell: if its RMS is 1e-3 of the leading term, it
# cannot matter, and no amount of extra compute will change that.
# -----------------------------------------------------------------------------
def report_residual_terms(params, cfg, data, n=20000, verbose=True):
    c = data["colloc"]
    k = min(n, c["n"])
    sl = slice(0, k)
    out = swe_residuals(params, cfg, data,
                        c["tn"][sl], c["xn"][sl], c["yn"][sl],
                        return_terms=True)
    rms = {k2: float(jnp.sqrt(jnp.mean(v ** 2)))
           for k2, v in out["terms"].items()}
    mom = {k2: v for k2, v in rms.items() if not k2.startswith("mass")}
    lead = max(mom.values()) if mom else 1.0
    share = {k2: v / lead for k2, v in mom.items()}

    res = dict(term_rms=rms, term_share_of_leading=share,
               R_mass_rms=float(jnp.sqrt(jnp.mean(out["R_mass"] ** 2))),
               R_u_rms=float(jnp.sqrt(jnp.mean(out["R_u"] ** 2))),
               R_v_rms=float(jnp.sqrt(jnp.mean(out["R_v"] ** 2))),
               derivative_calls=derivative_call_count())
    if verbose:
        print("\n[TERMS] u-momentum term magnitudes (RMS, non-dimensional):")
        for k2, v in sorted(mom.items(), key=lambda kv: -kv[1]):
            flag = ""
            if share[k2] < 1e-2:
                flag = "   <-- below 1% of leading: cannot influence the fit"
            print(f"    {k2:<15} {v:11.5f}  ({share[k2]*100:6.2f}%){flag}")
        print(f"  |R_mass|={res['R_mass_rms']:.4e}  "
              f"|R_u|={res['R_u_rms']:.4e}  |R_v|={res['R_v_rms']:.4e}")
    return res

In [8]:
# =============================================================================
# CELL 6 / 9  --  LOSS, CAUSAL WEIGHTING, GRADIENT-NORM BALANCING
# =============================================================================
# Three corrections to the V17 loss, all driven by the V17 telemetry:
#
# (1) GRADIENT-NORM BALANCING, not loss-value weighting.
#     V17 end of training (wind_on_diff_on, epoch 6000):
#         loss_data = 5.1e-6   loss_physics = 7.4e-6
#         loss_Cf_TV = 9.9e-3  loss_total  = 5.8e-4
#     so the total was ~95% TV -- while TV was simultaneously INCREASING
#     (0.0021 -> 0.0099), i.e. the field was roughening under a term that
#     dominated the objective. Loss values alone cannot diagnose that. Here
#     each term's gradient norm is measured periodically and the weights are
#     rescaled so TV can never exceed `tv_max_grad_share` of the total.
#
# (2) REAL CAUSAL WEIGHTING.
#     V17 used w(t) = exp(-eps*t), a static monotone decay that simply
#     down-weights late times. That is not the Wang, Sankaran & Perdikaris
#     (2024) scheme. Here time is chunked and chunk i is weighted by
#     exp(-eps * sum_{k<i} L_k) under stop_gradient, so a chunk is only
#     activated once its predecessors are solved.
#
# (3) SATURATION PENALTY on the friction pre-activation, to keep the branch in
#     its responsive region (see Cell 4).
#
# The phase indicator is computed from the TRACED epoch argument, never from a
# Python branch outside the compiled step -- otherwise the phase transition
# would be constant-folded away and leave no trace in the loss curve.
# =============================================================================

import jax
import jax.numpy as jnp


# -----------------------------------------------------------------------------
def phase_ramp(epoch, cfg):
    """Continuous 0 -> 1 ramp. Traced: safe inside jit."""
    c = cfg.cost
    return jnp.clip((epoch - c.phase1_epochs) / max(c.warm_epochs, 1), 0.0, 1.0)


def scheduled_weights(epoch, cfg, balance):
    """balance: dict of multiplicative corrections from gradient balancing."""
    r = phase_ramp(epoch, cfg)
    L = cfg.loss
    lerp = lambda pair: pair[0] + r * (pair[1] - pair[0])
    return dict(
        data=lerp(L.w_data) * balance["data"],
        swe=lerp(L.w_swe) * balance["swe"],
        tv=lerp(L.w_tv) * balance["tv"],
        bc=L.w_bc * balance["bc"],
        sat=cfg.arch.fric_saturation_penalty,
        ramp=r,
    )


def init_balance():
    return dict(data=1.0, swe=1.0, tv=1.0, bc=1.0)


# -----------------------------------------------------------------------------
def pseudo_huber(res, delta):
    return delta ** 2 * (jnp.sqrt(1.0 + (res / delta) ** 2) - 1.0)


def causal_weights(tn, sq_res, cfg):
    """Wang et al. (2024) causal chunk weights, stop-gradient."""
    M = cfg.loss.causal_chunks
    idx = jnp.clip(((tn + 1.0) * 0.5 * M).astype(jnp.int32), 0, M - 1)
    tot = jnp.zeros((M,), dtype=sq_res.dtype).at[idx].add(sq_res)
    cnt = jnp.zeros((M,), dtype=sq_res.dtype).at[idx].add(jnp.ones_like(sq_res))
    chunk_loss = tot / jnp.maximum(cnt, 1.0)
    cum = jnp.concatenate(
        [jnp.zeros((1,), dtype=chunk_loss.dtype), jnp.cumsum(chunk_loss)[:-1]])
    w = jnp.exp(-cfg.loss.causal_eps * cum)
    w = jax.lax.stop_gradient(w / jnp.maximum(w.max(), 1e-30))
    return w[idx], chunk_loss


# -----------------------------------------------------------------------------
def tv_regulariser(params, cfg, xn, yn):
    """Total variation of Cf, via exact JVPs of the friction branch."""
    f = lambda x, y: fric_forward(params, cfg, x, y)[0]        # noqa: F821
    ones = jnp.ones_like(xn)
    cf, cf_x = jax.jvp(lambda x: f(x, yn), (xn,), (ones,))
    _, cf_y = jax.jvp(lambda y: f(xn, y), (yn,), (ones,))
    # Normalised by the band width so the weight is dimensionless
    scale = cfg.phys.cf_hi - cfg.phys.cf_lo
    return jnp.mean(jnp.sqrt((cf_x / scale) ** 2 + (cf_y / scale) ** 2 + 1e-12))


# -----------------------------------------------------------------------------
def loss_components(params, cfg, data, batch, epoch, balance):
    w = scheduled_weights(epoch, cfg, balance)
    L = cfg.loss

    # ---- data term --------------------------------------------------------
    ob = batch["obs"]
    h, u, v = hydro_forward(params, cfg, ob["tn"], ob["xn"], ob["yn"])  # noqa
    res_h = h - ob["h"]
    loss_data = jnp.mean(pseudo_huber(res_h, L.huber_delta))
    if data["obs"]["has_uv"]:
        loss_data = loss_data + jnp.mean(
            pseudo_huber(u - ob["u"], L.huber_delta)
            + pseudo_huber(v - ob["v"], L.huber_delta))

    # ---- physics term -----------------------------------------------------
    cp = batch["colloc"]
    out = swe_residuals(params, cfg, data, cp["tn"], cp["xn"], cp["yn"])  # noqa
    sq = out["R_mass"] ** 2 + out["R_u"] ** 2 + out["R_v"] ** 2
    if L.causal_enabled:
        cw, chunk_loss = causal_weights(cp["tn"], jax.lax.stop_gradient(sq), cfg)
        loss_swe = jnp.mean(cw * sq)
    else:
        chunk_loss = jnp.zeros((L.causal_chunks,))
        loss_swe = jnp.mean(sq)

    # ---- regularisers -----------------------------------------------------
    loss_tv = tv_regulariser(params, cfg, cp["xn"], cp["yn"])
    loss_sat = jnp.mean(out["z"] ** 2)

    bc = batch["bc"]
    loss_bc = jnp.mean(boundary_residual(params, cfg, data, bc) ** 2)  # noqa

    total = (w["data"] * loss_data + w["swe"] * loss_swe
             + w["tv"] * loss_tv + w["bc"] * loss_bc + w["sat"] * loss_sat)

    aux = dict(loss_total=total, loss_data=loss_data, loss_swe=loss_swe,
               loss_tv=loss_tv, loss_bc=loss_bc, loss_sat=loss_sat,
               w_data=w["data"], w_swe=w["swe"], w_tv=w["tv"], ramp=w["ramp"],
               cf_mean=jnp.mean(out["cf"]), z_abs_mean=jnp.mean(jnp.abs(out["z"])),
               R_mass=jnp.sqrt(jnp.mean(out["R_mass"] ** 2)),
               R_u=jnp.sqrt(jnp.mean(out["R_u"] ** 2)),
               R_v=jnp.sqrt(jnp.mean(out["R_v"] ** 2)),
               causal_chunk_loss=chunk_loss)
    return total, aux


def loss_fn(params, cfg, data, batch, epoch, balance):
    return loss_components(params, cfg, data, batch, epoch, balance)


# -----------------------------------------------------------------------------
# Gradient-norm balancing. Called every `grad_balance_every` epochs OUTSIDE the
# compiled step, so it never affects the hot loop's compilation.
# -----------------------------------------------------------------------------
def _single_term_loss(name):
    def f(params, cfg, data, batch, epoch):
        L = cfg.loss
        if name == "data":
            ob = batch["obs"]
            h, u, v = hydro_forward(params, cfg, ob["tn"], ob["xn"], ob["yn"])  # noqa
            out = jnp.mean(pseudo_huber(h - ob["h"], L.huber_delta))
            if data["obs"]["has_uv"]:
                out = out + jnp.mean(pseudo_huber(u - ob["u"], L.huber_delta)
                                     + pseudo_huber(v - ob["v"], L.huber_delta))
            return out
        if name == "swe":
            cp = batch["colloc"]
            o = swe_residuals(params, cfg, data, cp["tn"], cp["xn"], cp["yn"])  # noqa
            return jnp.mean(o["R_mass"] ** 2 + o["R_u"] ** 2 + o["R_v"] ** 2)
        if name == "tv":
            cp = batch["colloc"]
            return tv_regulariser(params, cfg, cp["xn"], cp["yn"])
        if name == "bc":
            return jnp.mean(boundary_residual(params, cfg, data, batch["bc"]) ** 2)  # noqa
        raise ValueError(name)
    return f


def grad_norms(params, cfg, data, batch, epoch):
    out = {}
    for name in ("data", "swe", "tv", "bc"):
        g = jax.grad(_single_term_loss(name))(params, cfg, data, batch, epoch)
        leaves = jax.tree_util.tree_leaves(g)
        out[name] = float(jnp.sqrt(sum(jnp.sum(l ** 2) for l in leaves)))
    return out


def update_balance(balance, params, cfg, data, batch, epoch, verbose=False):
    """Rescale the TV weight so it cannot dominate, and equalise data vs swe."""
    L = cfg.loss
    if not L.grad_balance:
        return balance, {}
    w = scheduled_weights(epoch, cfg, balance)
    gn = grad_norms(params, cfg, data, batch, epoch)
    eff = {k: gn[k] * float(w[k]) for k in gn}
    tot = sum(eff.values()) + 1e-30
    share = {k: eff[k] / tot for k in eff}

    new = dict(balance)
    beta = L.grad_balance_beta
    # Cap the TV share
    if share["tv"] > L.tv_max_grad_share and eff["tv"] > 0:
        target = L.tv_max_grad_share / max(share["tv"], 1e-12)
        new["tv"] = beta * balance["tv"] + (1 - beta) * balance["tv"] * target
    # Keep the physics gradient within an order of magnitude of the data one
    if eff["swe"] > 0 and eff["data"] > 0:
        ratio = eff["data"] / eff["swe"]
        if ratio > 10.0 or ratio < 0.1:
            target = ratio ** 0.5
            new["swe"] = beta * balance["swe"] + (1 - beta) * balance["swe"] * target
    if verbose:
        print(f"    [balance] grad share: " + "  ".join(
            f"{k}={share[k]*100:.1f}%" for k in share) +
            f"   -> tv_mult={new['tv']:.3g} swe_mult={new['swe']:.3g}")
    return new, dict(grad_norms=gn, grad_share=share)

In [9]:
# =============================================================================
# CELL 7 / 9  --  TRAINING: ADAM (freeze-release) + L-BFGS-B (conditional)
# =============================================================================
# Adam is implemented inline rather than via optax, for three reasons:
#   * per-branch learning rates and an exact moment reset at the phase boundary
#     are trivial to express and impossible to get silently wrong;
#   * one fewer Kaggle image dependency;
#   * the parameter tree stays a plain dict, which scipy L-BFGS-B needs anyway.
#
# TWO CORRECTIONS TO THE V17 L-BFGS STAGE
#
# (a) DETERMINISTIC OBJECTIVE. A quasi-Newton method builds a curvature model
#     from successive gradients. If the collocation batch is resampled between
#     evaluations, that model is being fitted to noise, and the line search
#     will happily walk along a direction that is only flat for the current
#     sample. The V17 signature -- loss keeps falling while RMSE(Cf) degrades
#     -- is exactly what that produces, and it is the same mechanism Radfar
#     (2026) reports. Here L-BFGS sees ONE FIXED, LARGE point set for its
#     entire run.
#
# (b) HONEST CONVERGENCE REPORTING. V17 recorded nit=1000 and
#     converged=False in every single run while the proposal text promised
#     "natural convergence, not an arbitrary iteration cap". The scipy status,
#     message, nit, nfev and wall time are all recorded verbatim below, and the
#     stage is reverted if RMSE(Cf) worsens.
# =============================================================================

import time
import numpy as np
import jax
import jax.numpy as jnp
from jax.flatten_util import ravel_pytree


# -----------------------------------------------------------------------------
# Hand-rolled Adam with per-branch learning rates
# -----------------------------------------------------------------------------
def zeros_like_tree(tree):
    return jax.tree_util.tree_map(jnp.zeros_like, tree)


def init_adam_state(params):
    sub = {k: params[k] for k in trainable_keys()}                # noqa: F821
    return dict(m=zeros_like_tree(sub), v=zeros_like_tree(sub), step=0)


def reset_branch_moments(state, branch):
    st = dict(state)
    st["m"] = dict(st["m"]); st["v"] = dict(st["v"])
    st["m"][branch] = zeros_like_tree(st["m"][branch])
    st["v"][branch] = zeros_like_tree(st["v"][branch])
    return st


def lr_schedule(epoch, cfg):
    """Returns (lr_hydro, lr_fric). Traced-safe."""
    o, c = cfg.opt, cfg.cost
    prog = jnp.clip(epoch / max(c.epochs, 1), 0.0, 1.0)
    cos = o.lr_decay_to + (1.0 - o.lr_decay_to) * 0.5 * (1.0 + jnp.cos(jnp.pi * prog))
    lr_h = o.lr_hydro * cos

    if o.freeze_release:
        ramp = jnp.clip((epoch - c.phase1_epochs) / max(c.warm_epochs, 1), 0.0, 1.0)
    else:
        ramp = jnp.ones_like(prog)     # ablation baseline: single-stage training
    lr_f = o.lr_fric * cos * ramp
    return lr_h, lr_f


def adam_apply(params, grads, state, epoch, cfg,
               b1=0.9, b2=0.999, eps=1e-8):
    o = cfg.opt
    lr_h, lr_f = lr_schedule(epoch, cfg)
    lrs = dict(hydro=lr_h, fric=lr_f)
    step = state["step"] + 1

    # global gradient-norm clipping on the trainable subtree
    sub_g = {k: grads[k] for k in trainable_keys()}               # noqa: F821
    gnorm = jnp.sqrt(sum(jnp.sum(l ** 2)
                         for l in jax.tree_util.tree_leaves(sub_g)))
    scale = jnp.minimum(1.0, o.grad_clip / (gnorm + 1e-12))

    new_params = dict(params)
    new_m, new_v = dict(state["m"]), dict(state["v"])
    for branch in trainable_keys():                               # noqa: F821
        g = jax.tree_util.tree_map(lambda a: a * scale, grads[branch])
        m = jax.tree_util.tree_map(lambda mm, gg: b1 * mm + (1 - b1) * gg,
                                   state["m"][branch], g)
        v = jax.tree_util.tree_map(lambda vv, gg: b2 * vv + (1 - b2) * gg ** 2,
                                   state["v"][branch], g)
        mh = jax.tree_util.tree_map(lambda a: a / (1 - b1 ** step), m)
        vh = jax.tree_util.tree_map(lambda a: a / (1 - b2 ** step), v)
        new_params[branch] = jax.tree_util.tree_map(
            lambda p, a, b: p - lrs[branch] * a / (jnp.sqrt(b) + eps),
            params[branch], mh, vh)
        new_m[branch], new_v[branch] = m, v

    return new_params, dict(m=new_m, v=new_v, step=step), gnorm


# -----------------------------------------------------------------------------
# Batching
# -----------------------------------------------------------------------------
def make_batcher(cfg, data):
    c = cfg.cost
    n_obs, n_col, n_bc = data["obs"]["n"], data["colloc"]["n"], data["bc"]["n"]

    def draw(key):
        k1, k2, k3 = jax.random.split(key, 3)
        io = jax.random.randint(k1, (min(c.batch_data, n_obs),), 0, n_obs)
        ic = jax.random.randint(k2, (min(c.batch_phys, n_col),), 0, n_col)
        ib = jax.random.randint(k3, (min(n_bc, c.batch_phys // 4 or 1),), 0, n_bc)
        obs = {k: data["obs"][k][io] for k in ("tn", "xn", "yn", "h", "u", "v")}
        col = {k: data["colloc"][k][ic] for k in ("tn", "xn", "yn")}
        bc = {k: data["bc"][k][ib] for k in ("tn", "xn", "yn", "normal")}
        return dict(obs=obs, colloc=col, bc=bc)
    return draw


def make_train_step(cfg, data):
    """cfg and data are CLOSED OVER, so no static-argnum bookkeeping."""
    def step(params, state, batch, epoch, balance):
        (loss, aux), grads = jax.value_and_grad(
            loss_fn, has_aux=True)(params, cfg, data, batch, epoch, balance)  # noqa
        params, state, gnorm = adam_apply(params, grads, state, epoch, cfg)
        aux["grad_norm"] = gnorm
        return params, state, aux
    return jax.jit(step)


# -----------------------------------------------------------------------------
# Lightweight Cf metric, needed inside the L-BFGS gate
# -----------------------------------------------------------------------------
def cf_field(params, cfg, ev):
    XN, YN = jnp.meshgrid(ev["xn_grid"], ev["yn_grid"], indexing="ij")
    cf, _ = fric_forward(params, cfg, XN.ravel(), YN.ravel())     # noqa: F821
    return cf.reshape(XN.shape)


def cf_rmse_pct(params, cfg, ev):
    pred = cf_field(params, cfg, ev)
    true = ev["cf_true"]
    rng = float(true.max() - true.min())
    return float(jnp.sqrt(jnp.mean((pred - true) ** 2)) / max(rng, 1e-30) * 100.0)


# -----------------------------------------------------------------------------
# Main Adam loop
# -----------------------------------------------------------------------------
def train_adam(cfg, data, params, seed=0, verbose=True):
    c = cfg.cost
    step_fn = make_train_step(cfg, data)
    draw = make_batcher(cfg, data)
    state = init_adam_state(params)
    balance = init_balance()                                      # noqa: F821
    key = jax.random.PRNGKey(seed)

    history, t0, did_reset = [], time.time(), False
    for e in range(1, c.epochs + 1):
        # exact moment reset at the phase boundary
        if cfg.opt.freeze_release and e == c.phase1_epochs + 1 and not did_reset:
            state = reset_branch_moments(state, "fric")
            did_reset = True
            if verbose:
                print(f"  [phase] epoch {e}: friction moments reset, "
                      f"release ramp over {c.warm_epochs} epochs")

        key, sk = jax.random.split(key)
        batch = draw(sk)
        params, state, aux = step_fn(params, state, batch,
                                     jnp.float64(e), balance)

        if e % cfg.loss.grad_balance_every == 0:
            balance, bal_info = update_balance(                   # noqa: F821
                balance, params, cfg, data, batch, jnp.float64(e),
                verbose=(verbose and e % (cfg.loss.grad_balance_every * 20) == 0))

        if e % c.log_every == 0 or e == 1:
            rec = {k: float(v) for k, v in aux.items()
                   if jnp.ndim(v) == 0}
            rec["epoch"] = e
            sat = saturation_stats(params, cfg,                   # noqa: F821
                                   batch["colloc"]["xn"], batch["colloc"]["yn"])
            rec.update({k: float(v) for k, v in sat.items()})
            history.append(rec)

            if verbose and (e % c.eval_every == 0 or e == 1):
                rc = cf_rmse_pct(params, cfg, data["eval"])
                print(f"  e={e:>7} L={rec['loss_total']:.3e} "
                      f"data={rec['loss_data']:.2e} swe={rec['loss_swe']:.2e} "
                      f"tv={rec['loss_tv']:.2e} ramp={rec['ramp']:.2f} "
                      f"sat={rec['saturated_fraction']:.2f} "
                      f"dsig={rec['sigmoid_deriv_mean']:.3f} "
                      f"RMSE(Cf)={rc:.1f}%")

    wall = time.time() - t0
    if verbose:
        print(f"  [adam] {c.epochs} epochs in {wall:.1f}s "
              f"({wall/c.epochs*1e3:.2f} ms/epoch)")
    return params, dict(history=history, adam_seconds=wall,
                        balance=balance,
                        peak_vram_mb=vram_peak_mb(),              # noqa: F821
                        moment_reset_applied=did_reset)


# -----------------------------------------------------------------------------
# L-BFGS-B on a fixed point set
# -----------------------------------------------------------------------------
def train_lbfgs(cfg, data, params, verbose=True):
    from scipy.optimize import minimize
    o, c = cfg.opt, cfg.cost
    if not o.lbfgs_enabled:
        return params, dict(skipped=True)

    # ONE fixed subset for the whole run. Deterministic objective.
    n_col = min(4 * c.batch_phys, data["colloc"]["n"])
    n_obs = min(4 * c.batch_data, data["obs"]["n"])
    n_bc = data["bc"]["n"]
    batch = dict(
        obs={k: data["obs"][k][:n_obs] for k in ("tn", "xn", "yn", "h", "u", "v")},
        colloc={k: data["colloc"][k][:n_col] for k in ("tn", "xn", "yn")},
        bc={k: data["bc"][k][:n_bc] for k in ("tn", "xn", "yn", "normal")})

    epoch = jnp.float64(c.epochs)
    balance = init_balance()                                      # noqa: F821
    frozen = {k: params[k] for k in ("B_hydro", "B_fric")}
    trainable = {k: params[k] for k in trainable_keys()}          # noqa: F821
    flat0, unravel = ravel_pytree(trainable)

    obj_jit = jax.jit(lambda flat: jax.value_and_grad(
        lambda f: loss_fn({**frozen, **unravel(f)}, cfg, data, batch,   # noqa
                          epoch, balance)[0])(flat))

    def fun(x):
        l, g = obj_jit(jnp.asarray(x))
        return float(l), np.asarray(g, dtype=np.float64)

    rmse_before = cf_rmse_pct(params, cfg, data["eval"])
    t0 = time.time()
    res = minimize(fun, np.asarray(flat0, dtype=np.float64), jac=True,
                   method="L-BFGS-B",
                   options=dict(maxiter=c.lbfgs_maxiter, maxfun=o.lbfgs_maxfun,
                                ftol=o.lbfgs_ftol, gtol=o.lbfgs_gtol,
                                maxcor=50, disp=False))
    wall = time.time() - t0

    cand = dict(frozen); cand.update(unravel(jnp.asarray(res.x)))
    rmse_after = cf_rmse_pct(cand, cfg, data["eval"])

    reverted = False
    if (o.lbfgs_revert_if_cf_worse
            and rmse_after > rmse_before + o.lbfgs_revert_tolerance_pp):
        reverted = True
        out_params = params
    else:
        out_params = cand

    info = dict(skipped=False, lbfgs_seconds=wall,
                lbfgs_nit=int(res.nit), lbfgs_nfev=int(res.nfev),
                lbfgs_status=int(res.status), lbfgs_message=str(res.message),
                lbfgs_converged=bool(res.status == 0),
                lbfgs_hit_iteration_cap=bool(res.nit >= c.lbfgs_maxiter),
                lbfgs_final_loss=float(res.fun),
                cf_rmse_pct_before=rmse_before,
                cf_rmse_pct_after=rmse_after,
                cf_delta_pp=rmse_after - rmse_before,
                reverted_to_adam=reverted,
                n_points_fixed=dict(obs=n_obs, colloc=n_col, bc=n_bc))
    if verbose:
        print(f"  [lbfgs] {res.nit} iters, {wall:.1f}s, status={res.status} "
              f"({res.message})")
        print(f"  [lbfgs] RMSE(Cf) {rmse_before:.2f}% -> {rmse_after:.2f}% "
              f"({info['cf_delta_pp']:+.2f} pp)"
              f"{'  REVERTED to Adam weights' if reverted else ''}")
        if info["lbfgs_hit_iteration_cap"]:
            print("  [lbfgs] WARNING: hit the iteration cap. This is NOT "
                  "convergence; do not describe it as such.")
    return out_params, info

In [10]:
# =============================================================================
# CELL 8 / 9  --  EVALUATION
# =============================================================================
# Every metric is computed on ALL post-warmup snapshots (full trajectory under
# active tidal and wind forcing), never on a single instant.
#
# THE MOST IMPORTANT ADDITION: CONTROL BASELINES FOR H1b.
#   V17 reported corr(Cf, wind) falling from 0.720 (wind off) to 0.098 (wind
#   on) and read that as support for H1b. But in the same runs the spatial
#   skill of the recovered field collapsed, r_cf 0.464 -> 0.061. An
#   unstructured field is decorrelated from EVERYTHING. The metric was
#   satisfied for the wrong reason.
#   Here the same correlation is reported for four fields side by side:
#     * the recovered Cf
#     * the TRUE Cf                 <- the value H1b should be approaching
#     * a constant field            <- degenerate floor
#     * a random smooth field       <- what "no information" looks like
#   H1b is only supported if the recovered field's wind correlation approaches
#   the TRUE field's value WHILE its spatial skill r_cf stays high. If it
#   instead approaches the random field's value, the metric is measuring
#   failure, not success.
#
# Second addition: a two-zone detection score. RMSE against a continuous field
# is a demanding metric that conflates amplitude and pattern. The scientific
# claim the thesis actually needs ("sub-kilometre continuous resolution,
# unlike PG2017's two discrete zones") is better served by also reporting
# whether the rock/mud partition is recovered at all.
# =============================================================================

import dataclasses
import numpy as np
import jax
import jax.numpy as jnp


def _chunked_forward(params, cfg, tn, xn, yn, chunk=50000):
    hs, us, vs = [], [], []
    n = tn.size
    for i in range(0, n, chunk):
        sl = slice(i, min(i + chunk, n))
        h, u, v = hydro_forward(params, cfg, tn[sl], xn[sl], yn[sl])  # noqa
        hs.append(h); us.append(u); vs.append(v)
    return (jnp.concatenate(hs), jnp.concatenate(us), jnp.concatenate(vs))


def _pearson(a, b):
    a = np.asarray(a).ravel(); b = np.asarray(b).ravel()
    a = a - a.mean(); b = b - b.mean()
    den = np.sqrt((a ** 2).sum() * (b ** 2).sum())
    return float((a * b).sum() / den) if den > 0 else float("nan")


def mean_wind_stress_field(cfg, ref):
    """Time-mean |tau| on cell centres, over the evaluated window."""
    X, Y = np.meshgrid(ref["xc"], ref["yc"], indexing="ij")
    acc = np.zeros_like(X)
    for t in ref["t"]:
        tx, ty = wind_stress(t, X, Y, cfg, np)                    # noqa: F821
        acc += np.sqrt(np.asarray(tx) ** 2 + np.asarray(ty) ** 2)
    return acc / max(len(ref["t"]), 1)


def random_smooth_field(cfg, ref, seed=12345):
    """A field with no information about Cf but a comparable smoothness, used
    as the null control for the wind-correlation metric."""
    rng = np.random.default_rng(seed)
    nx, ny = cfg.domain.nx, cfg.domain.ny
    f = rng.normal(size=(nx, ny))
    # crude Gaussian smoothing by repeated box averaging
    for _ in range(12):
        f = 0.25 * (np.roll(f, 1, 0) + np.roll(f, -1, 0)
                    + np.roll(f, 1, 1) + np.roll(f, -1, 1))
    f = (f - f.mean()) / (f.std() + 1e-30)
    p = cfg.phys
    mid = 0.5 * (p.cf_rock + p.cf_mud)
    amp = 0.5 * (p.cf_rock - p.cf_mud)
    return np.clip(mid + amp * f, p.cf_lo, p.cf_hi)


def two_zone_scores(cf_pred, cf_true, cfg):
    """How well is the rock/mud partition recovered, independent of amplitude?"""
    p = cfg.phys
    thr_true = 0.5 * (p.cf_rock + p.cf_mud)
    y = (np.asarray(cf_true) > thr_true).ravel().astype(int)
    s = np.asarray(cf_pred).ravel()
    if y.min() == y.max():
        return dict(zone_auc=float("nan"), zone_best_accuracy=float("nan"),
                    zone_positive_fraction=float(y.mean()))
    # rank-based AUC
    order = np.argsort(s)
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, s.size + 1)
    n1 = y.sum(); n0 = y.size - n1
    auc = (ranks[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)
    # best achievable accuracy over all thresholds
    ss = np.sort(np.unique(s))
    cand = 0.5 * (ss[:-1] + ss[1:]) if ss.size > 1 else ss
    cand = cand[:: max(1, cand.size // 400)]
    acc = max(float(((s > t).astype(int) == y).mean()) for t in cand)
    return dict(zone_auc=float(auc), zone_best_accuracy=acc,
                zone_positive_fraction=float(y.mean()))


# -----------------------------------------------------------------------------
def evaluate(cfg, data, ref, params, extra=None, verbose=True):
    ev = data["eval"]
    s = cfg.scl
    ns, nx, ny = ev["shape"]

    h, u, v = _chunked_forward(params, cfg, ev["tn"], ev["xn"], ev["yn"])
    h = np.asarray(h).reshape(ns, nx, ny) * s.H_eta
    u = np.asarray(u).reshape(ns, nx, ny) * s.U
    v = np.asarray(v).reshape(ns, nx, ny) * s.U

    h_t, u_t, v_t = ref["eta"], ref["u"], ref["v"]
    D_true = ref["H"][None] + h_t
    shallow = D_true < 40.0

    def rng_of(a): return float(a.max() - a.min())

    res = dict(
        case=f"Wind={'ON' if cfg.wind_enabled else 'OFF'} | "
             f"Diffusion={'ON' if cfg.diffusion_enabled else 'OFF'}",
        tag=cfg.tag(), run_mode=cfg.run_mode,
        fingerprint=cfg.fingerprint(),
        n_eval_snapshots=int(ns),
        decoupled_collocation=True,
        total_points_data=int(data["obs"]["n"]),
        total_points_physics=int(data["colloc"]["n"]),
        effective_passes_physics=float(cfg.cost.effective_passes_physics),
        observed_variables=("h,u,v" if data["obs"]["has_uv"] else "h"),
    )

    # ---- reference dynamic range, reported BEFORE any percentage ----------
    res.update(ref_u_range=rng_of(u_t), ref_v_range=rng_of(v_t),
               ref_speed_mean=float(np.sqrt(u_t ** 2 + v_t ** 2).mean()),
               ref_eta_range=rng_of(h_t))

    # ---- state errors, absolute AND relative -----------------------------
    res.update(
        mae_h=float(np.abs(h - h_t).mean()),
        mae_u=float(np.abs(u - u_t).mean()),
        mae_v=float(np.abs(v - v_t).mean()),
        mae_h_pct=float(np.abs(h - h_t).mean() / max(rng_of(h_t), 1e-30) * 100),
        mae_u_pct=float(np.abs(u - u_t).mean() / max(rng_of(u_t), 1e-30) * 100),
        mae_v_pct=float(np.abs(v - v_t).mean() / max(rng_of(v_t), 1e-30) * 100),
        mae_u_pct_shallow=float(np.abs((u - u_t)[shallow]).mean()
                                / max(rng_of(u_t[shallow]), 1e-30) * 100),
        mae_v_pct_shallow=float(np.abs((v - v_t)[shallow]).mean()
                                / max(rng_of(v_t[shallow]), 1e-30) * 100),
    )

    # ---- friction field ---------------------------------------------------
    cf_pred = np.asarray(cf_field(params, cfg, ev))               # noqa: F821
    cf_true = np.asarray(ref["cf_true"])
    cf_rng = rng_of(cf_true)
    res.update(
        rmse_cf_abs=float(np.sqrt(((cf_pred - cf_true) ** 2).mean())),
        rmse_cf_pct=float(np.sqrt(((cf_pred - cf_true) ** 2).mean())
                          / max(cf_rng, 1e-30) * 100),
        bias_cf=float((cf_pred - cf_true).mean()),
        r_cf=_pearson(cf_pred, cf_true),
        cf_pred_min=float(cf_pred.min()), cf_pred_max=float(cf_pred.max()),
        cf_true_min=float(cf_true.min()), cf_true_max=float(cf_true.max()),
    )
    res.update(two_zone_scores(cf_pred, cf_true, cfg))

    # ---- H1b with controls ------------------------------------------------
    tau_bar = mean_wind_stress_field(cfg, ref)
    if tau_bar.std() < 1e-30:      # wind disabled -> use the ON field instead,
        cfg_on = dataclasses.replace(cfg, wind_enabled=True)      # noqa: F821
        tau_bar = mean_wind_stress_field(cfg_on, ref)
    rand = random_smooth_field(cfg, ref)
    res.update(
        corr_wind_recovered=_pearson(cf_pred, tau_bar),
        corr_wind_truth=_pearson(cf_true, tau_bar),
        corr_wind_random_control=_pearson(rand, tau_bar),
        corr_wind_constant_control=0.0,
    )
    # H1b is only supported if the recovered field's wind correlation moves
    # TOWARD the truth's value while r_cf stays high.
    d_truth = abs(res["corr_wind_recovered"] - res["corr_wind_truth"])
    d_rand = abs(res["corr_wind_recovered"] - res["corr_wind_random_control"])
    res["h1b_closer_to_truth_than_noise"] = bool(d_truth < d_rand)

    # ---- bathymetric-morphology relation (research question 5) ------------
    slope = np.sqrt(np.gradient(ref["b"], cfg.domain.dx, axis=0) ** 2
                    + np.gradient(ref["b"], cfg.domain.dy, axis=1) ** 2)
    res.update(corr_slope_recovered=_pearson(cf_pred, slope),
               corr_slope_truth=_pearson(cf_true, slope))

    # ---- residual term audit ---------------------------------------------
    try:
        res["terms"] = report_residual_terms(params, cfg, data,      # noqa
                                             verbose=False)
    except Exception as exc:
        res["terms"] = {"error": str(exc)}

    res["peak_vram_mb"] = vram_peak_mb()                             # noqa
    res["derivative_calls"] = derivative_call_count()                # noqa
    if extra:
        res.update(extra)

    # ---- success gate -----------------------------------------------------
    res["meets_rmse_target"] = bool(res["rmse_cf_pct"] < 5.0)
    res["meets_r_target"] = bool(res["r_cf"] > 0.95)
    res["meets_h1a_target"] = bool(res["mae_u_pct_shallow"] < 15.0)
    res["h1a_scale_valid"] = bool(0.03 <= res["ref_speed_mean"] <= 0.60)

    if verbose:
        print_evaluation(res)
    return res


def print_evaluation(r):
    print("\n" + "=" * 74)
    print(f"EVALUATION  {r['case']}   [{r['tag']}]  fp={r['fingerprint']}")
    print("=" * 74)
    print(f"reference scale   : mean speed {r['ref_speed_mean']:.3f} m/s, "
          f"u range {r['ref_u_range']:.3f} m/s, eta range {r['ref_eta_range']:.3f} m")
    print(f"                    scale plausible (VB25): "
          f"{'YES' if r['h1a_scale_valid'] else 'NO -- percentages below are meaningless'}")
    print(f"observed          : {r['observed_variables']}   "
          f"snapshots {r['n_eval_snapshots']}   "
          f"physics passes {r['effective_passes_physics']:.0f}")
    print("-" * 74)
    print(f"H1a  |u| error    : {r['mae_u']:.4f} m/s  "
          f"({r['mae_u_pct_shallow']:.2f}% of shallow range)  "
          f"target <15%  -> {'PASS' if r['meets_h1a_target'] else 'FAIL'}")
    print(f"     |v| error    : {r['mae_v']:.4f} m/s  ({r['mae_v_pct_shallow']:.2f}%)")
    print(f"     |h| error    : {r['mae_h']:.4f} m    ({r['mae_h_pct']:.2f}%)")
    print("-" * 74)
    print(f"H1   RMSE(Cf)     : {r['rmse_cf_pct']:.2f}% of range   "
          f"target <5%  -> {'PASS' if r['meets_rmse_target'] else 'FAIL'}")
    print(f"     r(Cf)        : {r['r_cf']:.3f}            "
          f"target >0.95 -> {'PASS' if r['meets_r_target'] else 'FAIL'}")
    print(f"     Cf bias      : {r['bias_cf']:+.5f}   "
          f"predicted range [{r['cf_pred_min']:.5f}, {r['cf_pred_max']:.5f}] "
          f"vs true [{r['cf_true_min']:.5f}, {r['cf_true_max']:.5f}]")
    print(f"     two-zone AUC : {r['zone_auc']:.3f}   "
          f"best accuracy {r['zone_best_accuracy']:.3f}")
    print("-" * 74)
    print("H1b  corr(Cf, mean wind stress) -- WITH CONTROLS:")
    print(f"     recovered    : {r['corr_wind_recovered']:+.3f}")
    print(f"     TRUE field   : {r['corr_wind_truth']:+.3f}   <- the target value")
    print(f"     random field : {r['corr_wind_random_control']:+.3f}   <- null control")
    print(f"     verdict      : recovered is closer to "
          f"{'TRUTH' if r['h1b_closer_to_truth_than_noise'] else 'NOISE'}")
    print(f"     (a low wind correlation is only evidence for H1b if r(Cf) is "
          f"also high; here r(Cf)={r['r_cf']:.3f})")
    print("-" * 74)
    tr = r.get("terms", {})
    if isinstance(tr, dict) and "term_share_of_leading" in tr:
        weak = {k: v for k, v in tr["term_share_of_leading"].items() if v < 1e-2}
        if weak:
            print("momentum terms below 1% of the leading term (cannot "
                  "influence the fit):")
            for k, v in weak.items():
                print(f"     {k:<14} {v*100:.4f}%")
    print(f"peak VRAM         : {r['peak_vram_mb']}")
    print(f"derivative calls  : {r['derivative_calls']}")
    print("=" * 74 + "\n")

In [11]:
# =============================================================================
# CELL 9 / 9  --  ABLATION DRIVER AND GO / NO-GO REPORT
# =============================================================================
# ONE GROUND TRUTH. It is generated once, with wind ON and the full spatially
# varying viscosity ON, and every arm inverts against that same truth. Only the
# INVERSE MODEL changes between arms.
#
# This is the single most important structural fix relative to V17, where the
# epoch-50 loss differed by a factor of 158 between the wind-on and wind-off
# arms -- a signature that the truth itself had been regenerated per arm. If
# both the truth and the model lose the wind term, the arm is a different,
# self-consistent world and tests nothing about Forcing Aliasing.
#
# ARMS
#   A  wind_off_diff_off   inverse model has NO wind term  -> the actual H1b /
#                          Forcing-Aliasing test
#   B  wind_on_diff_off    wind term present, no adaptive diffusion
#   C  wind_on_diff_on     the full proposed configuration
#   D  C but freeze_release=False  -> isolates the contribution of the
#                          freeze-release protocol (Validity Boundary 21)
#   E  C but u,v ALSO observed     -> the identifiability control. Not part of
#                          the scientific claim. It answers the one question
#                          that decides whether this thesis is viable:
#                          is the failure to recover Cf an OPTIMISATION
#                          problem or a STRUCTURAL identifiability limit?
# =============================================================================

import os, json, pickle, time, dataclasses
import numpy as np
import jax


ARMS = {
    "A_wind_off_diff_off": dict(wind_enabled=False, diffusion_enabled=False),
    "B_wind_on_diff_off":  dict(wind_enabled=True,  diffusion_enabled=False),
    "C_wind_on_diff_on":   dict(wind_enabled=True,  diffusion_enabled=True),
    "D_no_freeze_release": dict(wind_enabled=True,  diffusion_enabled=True,
                                _opt=dict(freeze_release=False)),
    "E_observe_uv":        dict(wind_enabled=True,  diffusion_enabled=True,
                                _obs=dict(observe_velocity=True)),
}


def arm_config(base, spec):
    kw = {k: v for k, v in spec.items() if not k.startswith("_")}
    cfg = dataclasses.replace(base, **kw)
    if "_opt" in spec:
        cfg = dataclasses.replace(cfg, opt=dataclasses.replace(cfg.opt, **spec["_opt"]))
    if "_obs" in spec:
        cfg = dataclasses.replace(cfg, obs=dataclasses.replace(cfg.obs, **spec["_obs"]))
    return cfg


def run_arm(name, cfg, ref, out_dir, verbose=True):
    print("\n" + "#" * 74)
    print(f"# ARM {name}   tag={cfg.tag()}   fp={cfg.fingerprint()}")
    print("#" * 74)
    audit_config(cfg, verbose=False)                              # noqa: F821

    t0 = time.time()
    data = prepare_all(cfg, ref)                                  # noqa: F821
    params = init_params(cfg, jax.random.PRNGKey(cfg.seed))       # noqa: F821

    # pre-training term audit: catches an inert term before spending the compute
    report_residual_terms(params, cfg, data, verbose=verbose)     # noqa: F821

    params, adam_info = train_adam(cfg, data, params,             # noqa: F821
                                   seed=cfg.seed, verbose=verbose)
    adam_params = jax.tree_util.tree_map(lambda a: a, params)
    eval_adam = evaluate(cfg, data, ref, adam_params,             # noqa: F821
                         extra=dict(stage="adam"), verbose=verbose)

    params, lbfgs_info = train_lbfgs(cfg, data, params, verbose=verbose)  # noqa
    eval_final = evaluate(cfg, data, ref, params,                 # noqa: F821
                          extra=dict(stage="final"), verbose=verbose)

    wall = time.time() - t0
    rec = dict(arm=name, tag=cfg.tag(),
               eval_adam=eval_adam, eval_final=eval_final,
               adam=adam_info, lbfgs=lbfgs_info,
               wall_seconds=wall,
               config=dataclasses.asdict(cfg),
               fingerprint=cfg.fingerprint())

    base = os.path.join(out_dir, f"v18_{cfg.tag()}")
    with open(base + "_adam_weights.pkl", "wb") as fh:
        pickle.dump(jax.tree_util.tree_map(np.asarray, adam_params), fh)
    with open(base + "_final_weights.pkl", "wb") as fh:
        pickle.dump(jax.tree_util.tree_map(np.asarray, params), fh)
    with open(base + "_history.pkl", "wb") as fh:
        pickle.dump(dict(history=adam_info["history"],
                         lbfgs=lbfgs_info, config_fp=cfg.fingerprint()), fh)
    print(f"[ARM {name}] done in {wall/60:.1f} min -> {base}_*.pkl")
    return rec


# -----------------------------------------------------------------------------
def run_ablation(arms=None, out_dir=OUT_DIR, verbose=True):            # noqa: F821
    arms = arms or list(ARMS)

    # ---- ONE truth, generated with wind ON and diffusion ON --------------
    cfg_truth = dataclasses.replace(CFG, wind_enabled=True,            # noqa: F821
                                    diffusion_enabled=True)
    print("\n[TRUTH] generating the single shared ground truth "
          "(wind ON, viscosity ON) ...")
    ref = run_reference(cfg_truth, verbose=verbose)                    # noqa: F821
    scale = report_reference_scale(ref, cfg_truth, strict=True)        # noqa: F821
    pde = verify_pde_coeffs(cfg_truth, ref, strict=True)               # noqa: F821
    with open(os.path.join(out_dir, "v18_reference.pkl"), "wb") as fh:
        pickle.dump(dict(ref={k: v for k, v in ref.items()},
                         scale=scale, pde_check=pde,
                         truth_fingerprint=cfg_truth.fingerprint()), fh)

    results = dict(_truth=dict(scale=scale, pde_check=pde,
                               fingerprint=cfg_truth.fingerprint()))
    for name in arms:
        cfg = arm_config(CFG, ARMS[name])                              # noqa: F821
        try:
            results[name] = run_arm(name, cfg, ref, out_dir, verbose)
        except Exception as exc:
            import traceback; traceback.print_exc()
            results[name] = dict(arm=name, error=repr(exc))
        with open(os.path.join(out_dir, "v18_ablation_results.pkl"), "wb") as fh:
            pickle.dump(results, fh)
    go_no_go(results)
    return ref, results


# -----------------------------------------------------------------------------
def go_no_go(results, thresholds=None):
    """Turn the ablation into a decision, not a paragraph.

    Thresholds are deliberately separated into two classes:
      * PUBLICATION targets (the proposal's own 5% / 0.95) -- what a success
        would look like;
      * VIABILITY thresholds -- what has to be true for Phase 2 to be worth
        starting at all.
    Failing the first is a result. Failing the second is a redesign.
    """
    T = dict(viable_rmse_pct=25.0, viable_r=0.60, viable_zone_auc=0.75,
             obsuv_rmse_pct=15.0)
    T.update(thresholds or {})

    print("\n" + "=" * 74)
    print("GO / NO-GO REPORT")
    print("=" * 74)

    tr = results.get("_truth", {})
    sc = tr.get("scale", {})
    gate0 = bool(sc and sc.get("clip_events", 1) == 0
                 and 0.03 <= sc.get("speed_mean", 0) <= 0.60)
    print(f"[GATE 0] reference solver clean and physically scaled  : "
          f"{'PASS' if gate0 else 'FAIL'}")
    if sc:
        print(f"          clip events {sc.get('clip_events')}, "
              f"mean speed {sc.get('speed_mean', float('nan')):.3f} m/s "
              f"(observed ~0.15)")
    if not gate0:
        print("  -> STOP. Every percentage metric downstream is uninterpretable "
              "until the reference regime is realistic (Validity Boundary 25).")

    rows = []
    for name, r in results.items():
        if name.startswith("_") or "error" in r:
            continue
        e = r["eval_final"]
        rows.append((name, e))

    print("\n" + "-" * 74)
    print(f"{'arm':<22}{'RMSE(Cf)%':>11}{'r(Cf)':>8}{'zoneAUC':>9}"
          f"{'|u|% shal':>11}{'L-BFGS':>10}")
    print("-" * 74)
    for name, e in rows:
        lb = results[name]["lbfgs"]
        tagl = ("rev" if lb.get("reverted_to_adam") else
                ("cap" if lb.get("lbfgs_hit_iteration_cap") else
                 ("ok" if lb.get("lbfgs_converged") else "-")))
        print(f"{name:<22}{e['rmse_cf_pct']:>11.2f}{e['r_cf']:>8.3f}"
              f"{e['zone_auc']:>9.3f}{e['mae_u_pct_shallow']:>11.2f}{tagl:>10}")
    print("-" * 74)

    by = {n: e for n, e in rows}

    # ---- Gate 1: hidden-state inference ----------------------------------
    c = by.get("C_wind_on_diff_on")
    if c:
        g1 = c["meets_h1a_target"] and c["h1a_scale_valid"]
        print(f"\n[GATE 1] H1a hidden-state inference (u,v from h)       : "
              f"{'PASS' if g1 else 'FAIL'}")
        print(f"          |u| error {c['mae_u']:.3f} m/s = "
              f"{c['mae_u_pct_shallow']:.1f}% of a "
              f"{c['ref_u_range']:.2f} m/s range")

    # ---- Gate 2: is the Cf failure structural or optimisational? ----------
    e_arm = by.get("E_observe_uv")
    if e_arm and c:
        structural = e_arm["rmse_cf_pct"] > T["obsuv_rmse_pct"]
        print(f"\n[GATE 2] identifiability control (u,v OBSERVED)        : "
              f"RMSE(Cf)={e_arm['rmse_cf_pct']:.1f}%, r={e_arm['r_cf']:.3f}")
        if structural:
            print("  -> Cf is NOT recovered even with full state observation.")
            print("     The limit is STRUCTURAL, not an optimisation failure.")
            print("     ACTION: H1 must be reformulated. A spatially continuous")
            print("     Cf field is not identifiable in this regime. Credible")
            print("     fallbacks, in order of decreasing ambition:")
            print("       (i)  invert a low-dimensional Cf basis (a few zones or")
            print("            a handful of smooth modes) rather than a free field;")
            print("       (ii) target the two-zone partition and report AUC, which")
            print("            is still a strict improvement over PG2017's fixed")
            print("            zones because the boundary is learned, not assumed;")
            print("       (iii) add an independent observable that carries friction")
            print("            information -- tide-gauge phase lag, or HF-radar /")
            print("            ADCP velocity assimilated as data, not validation.")
        else:
            print("  -> Cf IS recovered once u,v are observed.")
            print("     The limit is an OPTIMISATION / information problem, not")
            print("     a structural one. ACTION: the h-only case is worth more")
            print("     compute, a longer freeze-release ramp, and a sigma_fric /")
            print("     lambda_TV sweep. Keep H1 as stated.")

    # ---- Gate 3: does wind injection help? -------------------------------
    a_arm, b_arm = by.get("A_wind_off_diff_off"), by.get("B_wind_on_diff_off")
    if a_arm and b_arm:
        better = b_arm["rmse_cf_pct"] < a_arm["rmse_cf_pct"]
        print(f"\n[GATE 3] H1b wind injection helps Cf recovery          : "
              f"{'PASS' if better else 'FAIL'}")
        print(f"          no wind term : RMSE {a_arm['rmse_cf_pct']:.1f}%, "
              f"r {a_arm['r_cf']:.3f}, corr(Cf,wind) "
              f"{a_arm['corr_wind_recovered']:+.3f}")
        print(f"          wind term    : RMSE {b_arm['rmse_cf_pct']:.1f}%, "
              f"r {b_arm['r_cf']:.3f}, corr(Cf,wind) "
              f"{b_arm['corr_wind_recovered']:+.3f}")
        print(f"          TRUE field   : corr(Cf,wind) "
              f"{b_arm['corr_wind_truth']:+.3f}   <- the value to approach")
        print(f"          null control : corr(Cf,wind) "
              f"{b_arm['corr_wind_random_control']:+.3f}")
        if not b_arm["h1b_closer_to_truth_than_noise"]:
            print("  -> WARNING: the recovered field's wind correlation sits")
            print("     closer to the NOISE control than to the truth. A low")
            print("     wind correlation here is evidence of an unstructured")
            print("     field, NOT of successful wind/bed separation. Do not")
            print("     report it as support for H1b. This is precisely the")
            print("     false positive present in the V17 numbers.")

    # ---- Gate 4: does freeze-release earn its place? ---------------------
    d_arm = by.get("D_no_freeze_release")
    if d_arm and c:
        helps = c["rmse_cf_pct"] < d_arm["rmse_cf_pct"]
        print(f"\n[GATE 4] freeze-release protocol contributes (VB21)    : "
              f"{'PASS' if helps else 'FAIL'}")
        print(f"          with    : RMSE {c['rmse_cf_pct']:.1f}%, r {c['r_cf']:.3f}")
        print(f"          without : RMSE {d_arm['rmse_cf_pct']:.1f}%, "
              f"r {d_arm['r_cf']:.3f}")
        if not helps:
            print("  -> Innovation 3 is not earning its place. Report it as a "
                  "negative result rather than a contribution.")

    # ---- Gate 5: adaptive diffusion -------------------------------------
    if b_arm and c:
        drmse = abs(c["rmse_cf_pct"] - b_arm["rmse_cf_pct"])
        share = None
        t = c.get("terms", {})
        if isinstance(t, dict):
            share = t.get("term_share_of_leading", {}).get("diffusion")
        print(f"\n[GATE 5] adaptive diffusion is detectable              : "
              f"{'PASS' if drmse > 1.0 else 'FAIL'}")
        print(f"          |Delta RMSE(Cf)| between arms B and C = {drmse:.3f} pp")
        if share is not None:
            print(f"          diffusion term RMS = {share*100:.4f}% of the "
                  f"leading momentum term")
        if drmse <= 1.0:
            print("  -> Innovation 5 is numerically undetectable, as in V17.")
            print("     This is PHYSICS, not a bug: nu ~ 10-100 m^2/s over a")
            print("     25 km half-span gives a diffusive timescale of ~100 days")
            print("     against a 12 h tidal timescale. Either drop the novelty")
            print("     claim, restrict it to steep-gradient sub-domains, or")
            print("     justify a far larger nu on physical grounds.")

    # ---- overall ---------------------------------------------------------
    if c:
        viable = (c["rmse_cf_pct"] < T["viable_rmse_pct"]
                  or c["zone_auc"] > T["viable_zone_auc"])
        publishable = c["meets_rmse_target"] and c["meets_r_target"]
        print("\n" + "=" * 74)
        print(f"PROPOSAL TARGET (RMSE<5%, r>0.95) : "
              f"{'MET' if publishable else 'NOT MET'}")
        print(f"VIABILITY FOR PHASE 2             : "
              f"{'GO' if viable else 'NO-GO'}   "
              f"(RMSE<{T['viable_rmse_pct']:.0f}% or zone AUC>"
              f"{T['viable_zone_auc']:.2f})")
        if not publishable and viable:
            print("  -> Proceed, but rewrite the success criterion in Section 8.2")
            print("     before Phase 2. A 5% / 0.95 target on a free continuous")
            print("     field is not supported by any run in this project and is")
            print("     in tension with Radfar (2026) on h-only observability.")
        print("=" * 74 + "\n")
    return results


# -----------------------------------------------------------------------------
if __name__ == "__main__":
    REF, RESULTS = run_ablation()


[TRUTH] generating the single shared ground truth (wind ON, viscosity ON) ...
  step       0/26827  t=  0.00 h  |eta|max=0.000 m  |u|max=0.000 m/s
  step    2682/26827  t=  3.73 h  |eta|max=0.690 m  |u|max=0.077 m/s
  step    5364/26827  t=  7.45 h  |eta|max=0.598 m  |u|max=0.391 m/s
  step    8046/26827  t= 11.18 h  |eta|max=0.497 m  |u|max=0.298 m/s
  step   10728/26827  t= 14.90 h  |eta|max=1.055 m  |u|max=0.217 m/s
  step   13410/26827  t= 18.62 h  |eta|max=0.389 m  |u|max=0.549 m/s
  step   16092/26827  t= 22.35 h  |eta|max=1.126 m  |u|max=0.311 m/s
  step   18774/26827  t= 26.07 h  |eta|max=0.900 m  |u|max=0.189 m/s
  step   21456/26827  t= 29.80 h  |eta|max=0.462 m  |u|max=0.397 m/s
  step   24138/26827  t= 33.52 h  |eta|max=1.131 m  |u|max=0.351 m/s
  step   26820/26827  t= 37.25 h  |eta|max=0.394 m  |u|max=0.231 m/s

[REF-SCALE] --- Validity Boundary 25 audit ---
  clip events                     : 0
  min water depth seen            : 13.82 m
  u  min / max / range          

/tmp/ipykernel_58/310250873.py:240: DeprecationWarning: scipy.optimize: The `disp` and `iprint` options of the L-BFGS-B solver are deprecated and will be removed in SciPy 1.18.0.
  res = minimize(fun, np.asarray(flat0, dtype=np.float64), jac=True,


  [lbfgs] 2 iters, 11.9s, status=0 (CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH)
  [lbfgs] RMSE(Cf) 40.61% -> 40.61% (+0.00 pp)

EVALUATION  Wind=OFF | Diffusion=OFF   [wind_off_diff_off_smoke]  fp=33c1c7cf3b74
reference scale   : mean speed 0.080 m/s, u range 0.859 m/s, eta range 2.348 m
                    scale plausible (VB25): YES
observed          : h   snapshots 60   physics passes 100
--------------------------------------------------------------------------
H1a  |u| error    : 0.0794 m/s  (11.10% of shallow range)  target <15%  -> PASS
     |v| error    : 0.0147 m/s  (3.56%)
     |h| error    : 0.5099 m    (21.72%)
--------------------------------------------------------------------------
H1   RMSE(Cf)     : 40.61% of range   target <5%  -> FAIL
     r(Cf)        : -0.007            target >0.95 -> FAIL
     Cf bias      : -0.00040   predicted range [0.00346, 0.00346] vs true [0.00200, 0.00700]
     two-zone AUC : 0.496   best accuracy 0.637
--------------------------